# Domain III: CIFAR-100 - Stage III Scalability Validation

This notebook implements the CIFAR-100 validation domain (§5.4, §7.3) of the Informational Buildup Framework (IBF). CIFAR-100 tests whether the same correction dynamics remain **stable and non-destructive** in a higher-dimensional continual learning benchmark (d_eff = 28.9) built on frozen visual representations.

---

## Predictions Tested (§6.4)

**P7. Near-Zero Forgetting at Scale:** IBF achieves substantially lower backward-transfer degradation than all conventional baselines across 20 sequential tasks, with σ determined by geometric calibration rather than search.

**P8. Regime-Dependent Agency:** The agency channel produces negligible effect in CIFAR's saturated regime, in contrast to its essential role in chess and its harmful effect in RRW. The same mechanism produces three qualitatively different outcomes across three D-signal regimes.

---

## Domain Setup

| Component | Configuration |
|---|---|
| Frozen encoder | ViT-B/16 (ImageNet) → unsupervised PCA → 64D |
| Value corrections | 68D class-augmented space (64D image + 4D class features) |
| Agency channel | 64D image space |
| Task structure | 20 sequential tasks × 5 classes (Split-CIFAR-100) |
| Geometric calibration | σ = 5.0, d_eff = 28.9, κ = 0.93 |
| Seeds | 42 (primary), 123, 7 |

---

## ⚠ Log-Space Readout

**Table 3 in the paper reports log-space Task-IL accuracy:** `argmax_c [ log R_field(c) + δR(c) ]`

This notebook prints **both** linear and log-space metrics side by side. The mapping is:

| Notebook column | Paper table | Typical value |
|---|---|---|
| `avg(log)` / `acc_log` | **Table 3 Task-IL** | ≈ 0.892 |
| `avg(lin)` / `acc_lin` | *Not reported* | ≈ 0.839 |

If you see 0.839 in the output and expect 0.892 from the paper — check which column you're reading.

---

## Notebook Structure

| Cell | Role |
|---|---|
| C1–C4c | Setup, data pipeline, frozen encoder (ViT+PCA), coherence head, class-augmented encoding |
| C5 | σ calibration and κ measurement |
| C6–C6b | IBF engine (unchanged from chess) + CIFAR adapter layer |
| Smoke tests | D-signal health, coherence head separation, runtime estimate |
| C7 | Baselines: MLP, Replay (5000-sample buffer), EWC (λ=1000) |
| Overnight cell | Full IBF (3 seeds) + No-Agency ablation + all baselines |
| Ablation cell | No-Crucible + No-Crystallization |
| Weak-head cell | Deliberately weakened coherence head (2/class → 72.6% baseline) |
| Class-IL cell | 100-way evaluation without task oracle |
| Per-task cell | Correction scaling analysis vs baseline gap |

In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C1 — SETUP & DEPENDENCIES                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
!pip install scikit-learn -q

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import os
import json
import random
import copy
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from sklearn.decomposition import PCA

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

WORK_DIR = '/workspace'
DATA_DIR = os.path.join(WORK_DIR, 'data')
OUT_DIR = os.path.join(WORK_DIR, 'cifar_out')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Work: {WORK_DIR}, Data: {DATA_DIR}, Output: {OUT_DIR}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

Device: cuda
Work: /workspace, Data: /workspace/data, Output: /workspace/cifar_out


In [4]:
Z_DIM = 64
Z_DIM_AUG = 68
N_TASKS = 20
CLASSES_PER_TASK = 5
N_CLASSES = 100
EPOCHS_PER_TASK = 50
BATCH_SIZE = 32
MOVE_SCALE = 25.0

COH_PRETRAIN_PER_CLASS = 10   # was 50 — weaker head for stronger D signals
COH_PRETRAIN_EPOCHS = 10      # was 20

MAX_WRONG_CLASSES = None
RESET_D_CENTERING_PER_TASK = True

print(f"Config: {N_TASKS} tasks × {CLASSES_PER_TASK} classes, "
      f"{EPOCHS_PER_TASK} epochs/task, z_dim={Z_DIM}→{Z_DIM_AUG}")
print(f"  COH_PRETRAIN: {COH_PRETRAIN_PER_CLASS}/class × {COH_PRETRAIN_EPOCHS} epochs")
print(f"  MAX_WRONG_CLASSES={MAX_WRONG_CLASSES}, "
      f"RESET_D_CENTERING={RESET_D_CENTERING_PER_TASK}")

Config: 20 tasks × 5 classes, 50 epochs/task, z_dim=64→68
  COH_PRETRAIN: 10/class × 10 epochs
  MAX_WRONG_CLASSES=None, RESET_D_CENTERING=True


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C3 — DATA PIPELINE                                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("=" * 70)
print("  CELL C3 — DATA PIPELINE")
print("=" * 70)

transform_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

transform_vit_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

cifar100_train = torchvision.datasets.CIFAR100(
    root=DATA_DIR, train=True, download=True, transform=transform_vit_train)
cifar100_test = torchvision.datasets.CIFAR100(
    root=DATA_DIR, train=False, download=True, transform=transform_vit)

# Deterministic class ordering (shuffled once with fixed seed)
rng_split = random.Random(SEED)
class_order = list(range(N_CLASSES))
rng_split.shuffle(class_order)

tasks = []
for t in range(N_TASKS):
    tc = class_order[t * CLASSES_PER_TASK : (t + 1) * CLASSES_PER_TASK]
    tasks.append(tc)
    print(f"  Task {t:2d}: classes {tc}")

# Build per-class index maps (avoids scanning full dataset repeatedly)
train_indices_by_class = {c: [] for c in range(N_CLASSES)}
for i in range(len(cifar100_train)):
    _, label = cifar100_train[i]
    train_indices_by_class[label].append(i)

test_indices_by_class = {c: [] for c in range(N_CLASSES)}
for i in range(len(cifar100_test)):
    _, label = cifar100_test[i]
    test_indices_by_class[label].append(i)

def get_task_indices(indices_by_class, task_classes):
    idx = []
    for c in task_classes:
        idx.extend(indices_by_class[c])
    return idx

task_train_indices = [get_task_indices(train_indices_by_class, tc) for tc in tasks]
task_test_indices = [get_task_indices(test_indices_by_class, tc) for tc in tasks]

for t in range(N_TASKS):
    print(f"  Task {t:2d}: {len(task_train_indices[t])} train, "
          f"{len(task_test_indices[t])} test")

  CELL C3 — DATA PIPELINE
  Task  0: classes [42, 41, 91, 9, 65]
  Task  1: classes [50, 1, 70, 15, 78]
  Task  2: classes [73, 10, 55, 56, 72]
  Task  3: classes [45, 48, 92, 76, 37]
  Task  4: classes [30, 21, 32, 96, 80]
  Task  5: classes [49, 83, 26, 87, 33]
  Task  6: classes [8, 47, 59, 63, 74]
  Task  7: classes [44, 98, 52, 85, 12]
  Task  8: classes [36, 23, 39, 40, 18]
  Task  9: classes [66, 61, 60, 7, 34]
  Task 10: classes [99, 46, 2, 51, 16]
  Task 11: classes [38, 58, 68, 22, 62]
  Task 12: classes [24, 5, 6, 67, 82]
  Task 13: classes [19, 79, 43, 90, 20]
  Task 14: classes [0, 95, 57, 93, 53]
  Task 15: classes [89, 25, 71, 84, 77]
  Task 16: classes [64, 29, 27, 88, 97]
  Task 17: classes [4, 54, 75, 11, 69]
  Task 18: classes [86, 13, 17, 28, 31]
  Task 19: classes [35, 94, 3, 14, 81]
  Task  0: 2500 train, 500 test
  Task  1: 2500 train, 500 test
  Task  2: 2500 train, 500 test
  Task  3: 2500 train, 500 test
  Task  4: 2500 train, 500 test
  Task  5: 2500 train, 5

## Frozen Encoder — ViT-B/16 + Unsupervised PCA (§5.4)

The encoder is a frozen ViT-B/16 pretrained on ImageNet, projected via unsupervised PCA to a 64-dimensional image space. This is the supplied substrate — IBF operates on whatever geometry it provides.

The paper (§5.4): *"A frozen ViT-B/16 encoder pretrained on ImageNet is projected via unsupervised PCA to a 64-dimensional image space."*

All embeddings are pre-computed once (train and test) to avoid re-encoding every epoch. The slight train/test distribution shift from frozen augmentation is standard practice for CL benchmarks — all baselines share the same embeddings.

In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C4 — FROZEN ENCODER (ViT-B/16 + UNSUPERVISED PCA → 64D)        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C4 — FROZEN ENCODER")
print("=" * 70)

# - Step 1: Load frozen ViT-B/16 -
print("Loading ViT-B/16 (ImageNet pretrained)...")
vit = torchvision.models.vit_b_16(weights='IMAGENET1K_V1').to(DEVICE)
vit.eval()
for p in vit.parameters():
    p.requires_grad = False


class ViTFeatureExtractor(nn.Module):
    """Extract CLS token embeddings (768D) from frozen ViT."""
    def __init__(self, vit_model):
        super().__init__()
        self.vit = vit_model

    @torch.no_grad()
    def forward(self, x):
        x = self.vit._process_input(x)
        n = x.shape[0]
        cls_tok = self.vit.class_token.expand(n, -1, -1)
        x = torch.cat([cls_tok, x], dim=1)
        x = self.vit.encoder(x)
        return x[:, 0]  # CLS token → (batch, 768)


vit_encoder = ViTFeatureExtractor(vit).to(DEVICE)

# - Step 2: Extract 768D embeddings from ALL classes (for PCA fit) -

print("Extracting ViT embeddings for PCA fit (all 100 classes)...")

pca_loader = torch.utils.data.DataLoader(
    cifar100_test, batch_size=128, shuffle=False, num_workers=2)

all_embeddings_768 = []
all_labels_pca = []
with torch.no_grad():
    for imgs, lbls in pca_loader:
        feats = vit_encoder(imgs.to(DEVICE))
        all_embeddings_768.append(feats.cpu().numpy())
        all_labels_pca.append(lbls.numpy())

all_embeddings_768 = np.concatenate(all_embeddings_768, axis=0)  # (10000, 768)
all_labels_pca = np.concatenate(all_labels_pca, axis=0)
print(f"  Collected {len(all_embeddings_768)} embeddings, dim={all_embeddings_768.shape[1]}")

# - Step 3: Fit UNSUPERVISED PCA (768D → 64D)-
print(f"Fitting PCA: 768D → {Z_DIM}D (unsupervised, all classes)...")
pca_model = PCA(n_components=Z_DIM)
pca_model.fit(all_embeddings_768)

cumvar = np.cumsum(pca_model.explained_variance_ratio_)
print(f"  Variance explained by {Z_DIM} components: {cumvar[-1]:.4f}")
print(f"  First 10 components: {np.round(cumvar[:10], 3)}")

PCA_MEAN = pca_model.mean_.astype(np.float32)
PCA_COMPONENTS = pca_model.components_.astype(np.float32)  # (64, 768)

# - Step 4: Pre-compute ALL 64D embeddings (train + test) -

print("Pre-computing 64D embeddings for all train and test images...")


def encode_dataset_to_z64(dataset, batch_size=128):
    """Encode entire dataset → (N, 64) numpy array."""
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    z_all = []
    with torch.no_grad():
        for imgs, _ in loader:
            feats = vit_encoder(imgs.to(DEVICE)).cpu().numpy()
            z = (feats - PCA_MEAN) @ PCA_COMPONENTS.T
            z_all.append(z.astype(np.float32))
    return np.concatenate(z_all, axis=0)


# We need deterministic embeddings for evaluation, so re-encode test set
# through PCA (already done above, just project)
z64_test = ((all_embeddings_768 - PCA_MEAN) @ PCA_COMPONENTS.T).astype(np.float32)

# For training set, we encode once with the train transform.
print("  Encoding training set (single pass, frozen augmentation)...")
z64_train_all = encode_dataset_to_z64(cifar100_train)

# Extract labels as plain arrays
train_labels_all = np.array([cifar100_train[i][1] for i in range(len(cifar100_train))])
test_labels_all = np.array([cifar100_test[i][1] for i in range(len(cifar100_test))])

print(f"  Train: {z64_train_all.shape}, Test: {z64_test.shape}")
print(f"  z64 stats: mean={z64_train_all.mean():.3f}, "
      f"std={z64_train_all.std():.3f}, "
      f"range=[{z64_train_all.min():.3f}, {z64_train_all.max():.3f}]")

# Verify
test_z = z64_test[0]
print(f"  Sample z: shape={test_z.shape}, norm={np.linalg.norm(test_z):.3f}")


  CELL C4 — FROZEN ENCODER
Loading ViT-B/16 (ImageNet pretrained)...
Extracting ViT embeddings for PCA fit (all 100 classes)...
  Collected 10000 embeddings, dim=768
Fitting PCA: 768D → 64D (unsupervised, all classes)...
  Variance explained by 64 components: 0.5329
  First 10 components: [0.052 0.099 0.128 0.155 0.177 0.196 0.213 0.227 0.241 0.253]
Pre-computing 64D embeddings for all train and test images...
  Encoding training set (single pass, frozen augmentation)...
  Train: (50000, 64), Test: (10000, 64)
  z64 stats: mean=-0.004, std=1.579, range=[-12.782, 9.795]
  Sample z: shape=(64,), norm=12.636


## Coherence Head — The Baseline Evaluator (§5.4)

A modest 100-class classifier trained on a balanced sample from ALL classes, then frozen. This is the CIFAR analogue of chess's frozen win-probability head — it provides the initial coherence landscape that IBF's corrections build on top of.

The head is deliberately trained on only 10 samples/class (1000 total) to leave room for IBF corrections. At 10/class it reaches ~90% Task-IL accuracy, which means corrections are ceiling-limited in the standard configuration. The weak-head experiment (2/class, ~73% baseline) later tests whether corrections carry genuine signal when there's more room to improve.

In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C4b — COHERENCE HEAD (ALL 100 CLASSES, THEN FROZEN)             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C4b — COHERENCE HEAD")
print("=" * 70)

# Train a modest classifier on a balanced sample from ALL 100 classes.
# This gives a genuinely imperfect-but-universal baseline evaluator, 
# same role as the frozen win-probability head in chess.


class CoherenceHead(nn.Module):
    def __init__(self, z_dim=64, n_classes=100):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, z):
        return torch.softmax(self.head(z), dim=-1)


coherence_head = CoherenceHead(Z_DIM, N_CLASSES).to(DEVICE)
coh_opt = torch.optim.Adam(coherence_head.parameters(), lr=1e-3)

# Balanced pretrain set: COH_PRETRAIN_PER_CLASS images per class
pretrain_idx = []
for cls in range(N_CLASSES):
    cls_idx = train_indices_by_class[cls][:COH_PRETRAIN_PER_CLASS]
    pretrain_idx.extend(cls_idx)
random.shuffle(pretrain_idx)

pretrain_z = z64_train_all[pretrain_idx]
pretrain_y = train_labels_all[pretrain_idx]
print(f"  Pretraining on {len(pretrain_z)} samples "
      f"({COH_PRETRAIN_PER_CLASS}/class)")

for epoch in range(COH_PRETRAIN_EPOCHS):
    coherence_head.train()
    perm = np.random.permutation(len(pretrain_z))
    correct, total = 0, 0
    for start in range(0, len(pretrain_z), 128):
        batch_idx = perm[start:start + 128]
        z_b = torch.tensor(pretrain_z[batch_idx]).float().to(DEVICE)
        y_b = torch.tensor(pretrain_y[batch_idx]).long().to(DEVICE)
        probs = coherence_head(z_b)
        loss = nn.CrossEntropyLoss()(torch.log(probs + 1e-10), y_b)
        coh_opt.zero_grad()
        loss.backward()
        coh_opt.step()
        correct += (probs.argmax(1) == y_b).sum().item()
        total += len(y_b)
    if (epoch + 1) % 5 == 0:
        print(f"    Epoch {epoch+1}/{COH_PRETRAIN_EPOCHS}: "
              f"acc={correct/total:.3f}")

# Freeze coherence head
for p in coherence_head.parameters():
    p.requires_grad = False
coherence_head.eval()
del coh_opt

# Pre-compute coherence probabilities for all test images
print("  Pre-computing coherence probabilities (test set)...")
with torch.no_grad():
    coh_probs_test = []
    for start in range(0, len(z64_test), 256):
        z_b = torch.tensor(z64_test[start:start+256]).float().to(DEVICE)
        probs = coherence_head(z_b).cpu().numpy()
        coh_probs_test.append(probs)
    coh_probs_test = np.concatenate(coh_probs_test, axis=0)  # (10000, 100)

# Pre-compute for training set too
print("  Pre-computing coherence probabilities (train set)...")
with torch.no_grad():
    coh_probs_train = []
    for start in range(0, len(z64_train_all), 256):
        z_b = torch.tensor(z64_train_all[start:start+256]).float().to(DEVICE)
        probs = coherence_head(z_b).cpu().numpy()
        coh_probs_train.append(probs)
    coh_probs_train = np.concatenate(coh_probs_train, axis=0)  # (50000, 100)

# Verify
test_r = coh_probs_test[0, test_labels_all[0]]
print(f"  Sample R_field(class={test_labels_all[0]}): {test_r:.4f}")
print(f"  Coherence head frozen. Shape: {coh_probs_test.shape}")


  CELL C4b — COHERENCE HEAD
  Pretraining on 1000 samples (10/class)
    Epoch 5/10: acc=0.503
    Epoch 10/10: acc=0.706
  Pre-computing coherence probabilities (test set)...
  Pre-computing coherence probabilities (train set)...
  Sample R_field(class=49): 0.1293
  Coherence head frozen. Shape: (10000, 100)


In [8]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C4c — CLASS-AUGMENTED ENCODING (68D = 64D + 4D)                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C4c — CLASS-AUGMENTED ENCODING")
print("=" * 70)


def encode_class_features(class_id, n_total=N_CLASSES, scale=MOVE_SCALE):
    """4D class feature vector (scaled for sibling separation)."""
    return np.array([
        class_id / n_total,            # Global position
        (class_id % 10) / 10.0,        # Within-group position
        (class_id // 10) / 10.0,       # Group position
        float(class_id % 2),           # Parity
    ], dtype=np.float32) * scale


def encode_augmented(z_64, class_id):
    """Concatenate 64D image z + 4D class features → 68D."""
    return np.concatenate([z_64, encode_class_features(class_id)])


# Pre-compute class feature vectors for all 100 classes
CLASS_FEATURES = np.array([encode_class_features(c) for c in range(N_CLASSES)])

# Verify sibling separation
z_c0 = encode_augmented(z64_test[0], tasks[0][0])
z_c1 = encode_augmented(z64_test[0], tasks[0][1])
sib_dist = np.linalg.norm(z_c0 - z_c1)
print(f"  Sibling distance (same image, different class): {sib_dist:.3f}")
print(f"  Augmented z dim: {len(z_c0)} (expect {Z_DIM_AUG})")


  CELL C4c — CLASS-AUGMENTED ENCODING
  Sibling distance (same image, different class): 25.126
  Augmented z dim: 68 (expect 68)


## Geometric Resolution Calibration (§5.1)

Same calibration principle as RRW and chess: σ derived from latent geometry, no grid search.

The paper reports: d_eff = 28.9, σ = 5.0, κ = 0.93. The σ override to 5.0 enforces ≤1% kernel bleed at the P10 nearest-neighbor boundary in the 68D augmented space.

Agency operates in the underlying 64D image space with a companion scale.

In [29]:
print("=" * 70)
print("  CELL C5 — σ CALIBRATION")
print("=" * 70)

# 1. Calibration set
cal_z_64 = []
cal_labels = []
for cls in range(N_CLASSES):
    mask = test_labels_all == cls
    cls_z = z64_test[mask][:10]
    for z in cls_z:
        cal_z_64.append(z)
        cal_labels.append(cls)
cal_z_64 = np.array(cal_z_64)
cal_labels = np.array(cal_labels)
print(f"  Calibration: {len(cal_z_64)} samples (64D)")

# 2. d_eff
eigenvalues_64 = np.var(cal_z_64, axis=0)
eigenvalues_64 = np.sort(eigenvalues_64)[::-1]
d_eff = (np.sum(eigenvalues_64) ** 2) / np.sum(eigenvalues_64 ** 2)
cumvar_64 = np.cumsum(eigenvalues_64) / np.sum(eigenvalues_64)
d90 = int(np.searchsorted(cumvar_64, 0.90)) + 1
d95 = int(np.searchsorted(cumvar_64, 0.95)) + 1
print(f"  d_eff (participation ratio): {d_eff:.1f}")
print(f"  d90={d90}, d95={d95}")

# 3. 68D augmented vectors
cal_z_68 = np.array([encode_augmented(cal_z_64[i], cal_labels[i])
                      for i in range(len(cal_z_64))])

# 4. Distances
n_pairs = 10000
rng_cal = np.random.RandomState(SEED)
idx1 = rng_cal.randint(0, len(cal_z_68), n_pairs)
idx2 = rng_cal.randint(0, len(cal_z_68), n_pairs)
valid = idx1 != idx2
dists_68 = np.linalg.norm(cal_z_68[idx1[valid]] - cal_z_68[idx2[valid]], axis=1)
p10_68 = np.percentile(dists_68, 10)
p50_68 = np.median(dists_68)

# 5. Siblings
sib_dists = []
for i in range(min(200, len(cal_z_64))):
    c_true = cal_labels[i]
    c_other = (c_true + 1) % N_CLASSES
    z_true = encode_augmented(cal_z_64[i], c_true)
    z_other = encode_augmented(cal_z_64[i], c_other)
    sib_dists.append(np.linalg.norm(z_true - z_other))
sib_median = np.median(sib_dists)

# 6. σ OVERRIDE: σ=5 for strict geometric isolation.
# At σ=10, kernel bleed to nearest different image (d=17) was 24%.
# At σ=5, it's 0.3%. Prescribed by 1% bleed criterion at P10.
SIGMA_68D = 5.0
MERGE_68D = 7.5  # σ × 1.5

# Agency space: unchanged
dists_64d = np.linalg.norm(cal_z_64[idx1[valid][:5000]] -
                           cal_z_64[idx2[valid][:5000]], axis=1)
p10_64 = np.percentile(dists_64d, 10)
SIGMA_64D_AGENCY = 5.0  # Match value σ for tighter agency centers

print(f"\n  68D distances: P10={p10_68:.4f}, median={p50_68:.4f}")
print(f"  Sibling distances: median={sib_median:.4f}")
print(f"  SIGMA_68D (value)   = {SIGMA_68D:.4f}  [OVERRIDE: 1% bleed]")
print(f"  MERGE_68D           = {MERGE_68D:.4f}")
print(f"  SIGMA_64D (agency)  = {SIGMA_64D_AGENCY:.4f}")

kappa_cifar = SIGMA_68D / np.sqrt(d_eff)
print(f"\n  ╔═════════════════════════════════════════════════╗")
print(f"  ║  κ_cifar = {SIGMA_68D:.4f} / √{d_eff:.1f} = {kappa_cifar:.4f}   ║")
print(f"  ╚═════════════════════════════════════════════════╝")

# Geometric capacity
centers_est = [cal_z_68[0]]
for i in range(1, len(cal_z_68)):
    ds = np.linalg.norm(cal_z_68[i] - np.array(centers_est), axis=1)
    if np.all(ds >= MERGE_68D):
        centers_est.append(cal_z_68[i])
    if len(centers_est) > 5000:
        break
geo_capacity = len(centers_est)
print(f"  Geometric capacity: ~{geo_capacity} (need ≥100)")

# Bleed check
bleed_p10 = np.exp(-(p10_68**2) / (2 * SIGMA_68D**2))
bleed_sib = np.exp(-(sib_median**2) / (2 * SIGMA_68D**2))
print(f"  Kernel bleed at P10 ({p10_68:.1f}): {bleed_p10:.4f}")
print(f"  Kernel bleed at sibling ({sib_median:.1f}): {bleed_sib:.4f}")

C_RC = {
    'SIGMA': float(SIGMA_68D),
    'MERGE_THRESHOLD': float(MERGE_68D),
    'V_MAX': 0.50,
    'DR_CAPACITY': 50000,
}

  CELL C5 — σ CALIBRATION
  Calibration: 1000 samples (64D)
  d_eff (participation ratio): 28.9
  d90=48, d95=56

  68D distances: P10=19.9458, median=31.2603
  Sibling distances: median=25.1259
  SIGMA_68D (value)   = 5.0000  [OVERRIDE: 1% bleed]
  MERGE_68D           = 7.5000
  SIGMA_64D (agency)  = 5.0000

  ╔═════════════════════════════════════════════════╗
  ║  κ_cifar = 5.0000 / √28.9 = 0.9295   ║
  ╚═════════════════════════════════════════════════╝
  Geometric capacity: ~883 (need ≥100)
  Kernel bleed at P10 (19.9): 0.0004
  Kernel bleed at sibling (25.1): 0.0000


In [30]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C6 — IBF ENGINE                    (UNCHANGED from chess)          ║
# ║  Only the class definitions are included; the chess-specific smoke       ║
# ║  test is omitted (it depends on chess.Board which is not relevant).      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C6 — IBF ENGINE v4.1 (R19.6)")
print("=" * 70)


# - IBFParams, MemoryCenter, IBFEngine - VERBATIM from chess -

@dataclass
class IBFParams:
    sigma: float = 0.206
    merge_threshold: float = 0.366
    sigma_agency: float = None
    eta: float = 0.1
    eta_cryst: float = 0.005
    eta_k: float = 0.05
    mu_base: float = 0.06
    mu_crystallized: float = 0.001
    v_max: float = 0.50
    w_max: float = 5.0
    k_0: float = 5.0
    k_min: float = 1.0
    crystallization_threshold: int = 15
    convergence_threshold: float = 0.025
    n_cross_min: int = 8
    reversal_threshold: float = -0.0125
    w_dvar_threshold: float = 0.005
    activation_threshold: float = 0.01
    creation_threshold: float = 0.01
    capacity: int = 5000
    alpha_shrink: float = 100.0
    sigma_floor: float = 0.15
    min_samples_shrink: int = 50
    eps: float = 1e-6

    @classmethod
    def from_calibration(cls, C_RC_dict: dict):
        return cls(
            sigma=C_RC_dict['SIGMA'],
            merge_threshold=C_RC_dict['MERGE_THRESHOLD'],
            v_max=C_RC_dict.get('V_MAX', 0.50),
            capacity=C_RC_dict.get('DR_CAPACITY', 5000),
        )


@dataclass
class MemoryCenter:
    z: np.ndarray
    v: float = 0.0
    w: float = 0.0
    n_updates: int = 0
    D_sum: float = 0.0
    D_sq_sum: float = 0.0
    mu_eff: float = 0.06
    context_id: int = -1
    birth_epoch: int = 0
    context_update_counts: Dict[int, int] = field(default_factory=dict)
    sigma: float = 0.58
    D_history: List[float] = field(default_factory=list)
    D_history_cross: List[float] = field(default_factory=list)
    was_ever_crystallized: bool = False
    crucible_verified: bool = False
    dissolution_log: List[Dict] = field(default_factory=list)

    def is_crystallized(self):
        return self.mu_eff < 0.06 - 1e-6

    def D_var_rolling(self):
        if len(self.D_history) < 25:
            return 0.0
        rec = self.D_history[20:][-50:]
        if len(rec) < 5:
            return 0.0
        return float(np.var(rec))

    def n_cross_updates_total(self):
        return sum(cnt for ctx, cnt in self.context_update_counts.items()
                   if ctx != self.context_id)

    def clone(self):
        return MemoryCenter(
            z=self.z.copy(), v=self.v, w=self.w,
            n_updates=self.n_updates, D_sum=self.D_sum,
            D_sq_sum=self.D_sq_sum, mu_eff=self.mu_eff,
            context_id=self.context_id, birth_epoch=self.birth_epoch,
            context_update_counts=dict(self.context_update_counts),
            sigma=self.sigma,
            D_history=list(self.D_history),
            D_history_cross=list(self.D_history_cross),
            was_ever_crystallized=self.was_ever_crystallized,
            crucible_verified=self.crucible_verified,
            dissolution_log=list(self.dissolution_log),
        )


class IBFEngine:
    def __init__(self, params: IBFParams = None):
        self.p = params or IBFParams()
        self.value_centers: List[MemoryCenter] = []
        self.agency_centers: List[MemoryCenter] = []
        self.current_epoch = 0
        self.current_context_id = -1
        self._epoch_D_vals = []

        self._merge_ratio = self.p.merge_threshold / self.p.sigma
        self._dynamic_sigma_floor = min(self.p.sigma_floor,
                                        self.p.sigma * 0.25)
        self._needle_threshold = self.p.sigma * 0.50

        self._sigma_agency = (self.p.sigma_agency
                              if self.p.sigma_agency is not None
                              else self.p.sigma)

        self._D_running_sum = 0.0
        self._D_running_count = 0

        print(f"IBF Engine v4.1 (R19.6) initialized")
        print(f"  η={self.p.eta}, η_cryst={self.p.eta_cryst}, "
              f"μ_base={self.p.mu_base}")
        print(f"  η/μ transient={self.p.eta/self.p.mu_base:.2f}, "
              f"crystal={self.p.eta_cryst/self.p.mu_crystallized:.1f}")
        print(f"  k_0={self.p.k_0}, v_max={self.p.v_max}, "
              f"w_max={self.p.w_max}")
        print(f"  σ_value={self.p.sigma:.4f}, "
              f"σ_agency={self._sigma_agency:.4f}, "
              f"merge={self.p.merge_threshold:.4f}")
        print(f"  Convergence: |μ_D|<{self.p.convergence_threshold}, "
              f"n_min={self.p.crystallization_threshold}")
        print(f"  Crucible: n_cross_min={self.p.n_cross_min}, "
              f"rev_thresh={self.p.reversal_threshold}")
        print(f"  Shrink: α={self.p.alpha_shrink}, "
              f"σ_floor={self._dynamic_sigma_floor:.4f}")

    def kernel_batch(self, z, centers):
        if not centers:
            return np.array([])
        z_all = np.array([c.z for c in centers])
        diffs = z_all - z[np.newaxis, :]
        sq_dists = np.sum(diffs ** 2, axis=1)
        sigmas = np.array([c.sigma for c in centers])
        return np.exp(-sq_dists / (2.0 * sigmas ** 2))

    def _read_gate(self, c):
        if c.context_id == self.current_context_id:
            return 1.0
        if c.is_crystallized() and c.crucible_verified:
            return 1.0
        return 0.0

    def delta_R(self, z):
        if not self.value_centers:
            return 0.0
        K = self.kernel_batch(z, self.value_centers)
        total = 0.0
        for i, c in enumerate(self.value_centers):
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                total += g * c.v * K[i]
        return total

    def delta_R_batch(self, Z_mat):
        if not self.value_centers:
            return np.zeros(len(Z_mat))
        Z_c = np.array([c.z for c in self.value_centers])
        sigmas = np.array([c.sigma for c in self.value_centers])
        vs = np.array([c.v for c in self.value_centers])
        gate = np.array([self._read_gate(c) for c in self.value_centers])
        Z_sq = np.sum(Z_mat**2, axis=1, keepdims=True)
        C_sq = np.sum(Z_c**2, axis=1, keepdims=True).T
        sq_d = Z_sq + C_sq - 2.0 * (Z_mat @ Z_c.T)
        K = np.exp(-sq_d / (2.0 * sigmas[np.newaxis, :]**2))
        K[K < self.p.activation_threshold] = 0.0
        return K @ (gate * vs)

    def delta_k(self, z):
        if not self.agency_centers:
            return 0.0
        K = self.kernel_batch(z, self.agency_centers)
        total_w = 0.0
        sum_K = 0.0
        for i, c in enumerate(self.agency_centers):
            if not c.is_crystallized():
                continue
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                total_w += g * c.w * K[i]
                sum_K += g * K[i]
        return total_w / sum_K if sum_K > 1e-6 else 0.0

    def k_eff(self, z):
        return max(self.p.k_min, self.p.k_0 + self.delta_k(z))

    # - select_move - not used for CIFAR, included for completeness -

    def select_move(self, board, deterministic=False):
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return None, {}
        if len(legal_moves) == 1:
            return legal_moves[0], {'forced': True, 'k_pos': 0.0,
                                    'n_moves': 1}
        z_current = RC_encode(board)
        k_pos = self.k_eff(z_current)
        R_eff_current = RC_R_field(board)
        z_afters, r_fields = [], []
        for move in legal_moves:
            ba = apply_move(board, move)
            z_afters.append(RC_encode_move(board, ba, move))
            r_fields.append(RC_R_field(ba))
        Z_mat = np.array(z_afters)
        R_field_afters = np.array(r_fields)
        dR_afters = self.delta_R_batch(Z_mat)
        R_eff_afters = np.clip(1.0 - (R_field_afters + dR_afters), 0.0, 1.0)
        delta_R_j = R_eff_afters - R_eff_current
        logits = k_pos * delta_R_j
        logits -= np.max(logits)
        exp_logits = np.exp(logits)
        probs = exp_logits / np.sum(exp_logits)
        idx = (int(np.argmax(probs)) if deterministic
               else np.random.choice(len(legal_moves), p=probs))
        return legal_moves[idx], {
            'k_pos': float(k_pos), 'n_moves': len(legal_moves),
            'entropy': float(-np.sum(probs * np.log(probs + 1e-10))),
            'R_eff_current': float(R_eff_current),
            'R_eff_chosen': float(R_eff_afters[idx]),
            'delta_R_chosen': float(delta_R_j[idx]),
        }

    # - compute_D_and_update - the core learning step -

    def compute_D_and_update(self, board_before, board_after_own_move,
                             board_after_opponent,
                             move_chosen=None, move_opponent=None,
                             R_imposed_override=None):
        z_before = RC_encode(board_before)

        if move_chosen is not None:
            z_chosen = RC_encode_move(board_before, board_after_own_move,
                                     move_chosen)
        else:
            z_chosen = RC_encode(board_after_own_move)

        R_field_chosen = RC_R_field(board_after_own_move)
        dR_chosen = self.delta_R(z_chosen)
        R_eff_chosen = np.clip(1.0 - (R_field_chosen + dR_chosen),
                               0.0, 1.0)

        if R_imposed_override is not None:
            R_eff_imposed = float(R_imposed_override)
        else:
            if move_opponent is not None:
                z_revealed = RC_encode_move(board_after_own_move,
                                           board_after_opponent,
                                           move_opponent)
            else:
                z_revealed = RC_encode(board_after_opponent)
            R_field_imposed = RC_R_field(board_after_opponent)
            dR_revealed = self.delta_R(z_revealed)
            R_eff_imposed = np.clip(R_field_imposed + dR_revealed,
                                    0.0, 1.0)

        D = R_eff_imposed - R_eff_chosen

        self._D_running_count += 1
        self._D_running_sum += D
        D_mean = self._D_running_sum / self._D_running_count
        D = D - D_mean

        self._epoch_D_vals.append(D)
        self._update_value_channel(z_chosen, D)
        self._update_agency_channel(z_before, D)

        return {
            'D': D, 'skipped': False,
            'R_eff_chosen': float(R_eff_chosen),
            'R_eff_imposed': float(R_eff_imposed),
            'R_field_revealed': float(R_field_chosen),
            'dR_chosen': float(dR_chosen),
            'dR_revealed': 0.0,
        }

    def _thermodynamic_shrink(self, center):
        if len(center.D_history) >= self.p.min_samples_shrink:
            D_var = center.D_var_rolling()
            calc_sigma = max(self._dynamic_sigma_floor,
                             self.p.sigma / (1.0 + self.p.alpha_shrink
                                             * D_var))
            center.sigma = min(center.sigma, calc_sigma)

    def _update_value_channel(self, z_chosen, D):
        neg_D = -D
        cross_crystals = [c for c in self.value_centers
                          if c.is_crystallized()
                          and c.context_id != self.current_context_id]
        if cross_crystals:
            K_cross = self.kernel_batch(z_chosen, cross_crystals)
            for i, c in enumerate(cross_crystals):
                kw = float(K_cross[i])
                if kw < self.p.activation_threshold:
                    continue
                juris_D = neg_D * kw
                c.v = np.clip(c.v + self.p.eta_cryst * juris_D,
                              -self.p.v_max, self.p.v_max)
                c.n_updates += 1
                c.D_sum += juris_D
                c.D_sq_sum += juris_D * juris_D
                c.context_update_counts[self.current_context_id] = \
                    c.context_update_counts.get(
                        self.current_context_id, 0) + 1
                c.D_history_cross.append(neg_D)

        same_ctx = [c for c in self.value_centers
                    if c.context_id == self.current_context_id]
        if same_ctx:
            K_same = self.kernel_batch(z_chosen, same_ctx)
            max_K = float(np.max(K_same))
        else:
            K_same = np.array([])
            max_K = 0.0

        if max_K < self.p.creation_threshold:
            if len(self.value_centers) < self.p.capacity:
                juris_D = neg_D * 1.0
                nc = MemoryCenter(
                    z=z_chosen.copy(),
                    v=np.clip(self.p.eta * juris_D,
                              -self.p.v_max, self.p.v_max),
                    w=0.0, mu_eff=self.p.mu_base,
                    context_id=self.current_context_id,
                    birth_epoch=self.current_epoch,
                    sigma=self.p.sigma,
                )
                nc.n_updates = 1
                nc.D_sum = juris_D
                nc.D_sq_sum = juris_D * juris_D
                nc.D_history.append(juris_D)
                nc.context_update_counts[self.current_context_id] = 1
                self.value_centers.append(nc)
            return

        for i, c in enumerate(same_ctx):
            kw = float(K_same[i])
            if kw < self.p.activation_threshold:
                continue
            juris_D = neg_D * kw
            lr = (self.p.eta_cryst if c.is_crystallized()
                  else self.p.eta)
            c.v = np.clip(c.v + lr * juris_D,
                          -self.p.v_max, self.p.v_max)
            c.n_updates += 1
            c.D_sum += juris_D
            c.D_sq_sum += juris_D * juris_D
            c.context_update_counts[self.current_context_id] = \
                c.context_update_counts.get(
                    self.current_context_id, 0) + 1
            c.D_history.append(juris_D)
            self._thermodynamic_shrink(c)

    def _update_agency_channel(self, z_before, D):
        cross_crystals = [c for c in self.agency_centers
                          if c.is_crystallized()
                          and c.context_id != self.current_context_id]
        if cross_crystals:
            K_cross = self.kernel_batch(z_before, cross_crystals)
            for i, c in enumerate(cross_crystals):
                kw = float(K_cross[i])
                if kw < self.p.activation_threshold:
                    continue
                c.n_updates += 1
                c.context_update_counts[self.current_context_id] = \
                    c.context_update_counts.get(
                        self.current_context_id, 0) + 1
                c.D_history_cross.append(D)

        same_ctx = [c for c in self.agency_centers
                    if c.context_id == self.current_context_id]
        if same_ctx:
            K_same = self.kernel_batch(z_before, same_ctx)
            max_K = float(np.max(K_same))
        else:
            K_same = np.array([])
            max_K = 0.0

        if max_K < self.p.creation_threshold:
            if len(self.agency_centers) < self.p.capacity:
                nc = MemoryCenter(
                    z=z_before.copy(), v=0.0, w=0.0,
                    mu_eff=self.p.mu_base,
                    context_id=self.current_context_id,
                    birth_epoch=self.current_epoch,
                    sigma=self._sigma_agency,
                )
                juris_D = D * 1.0
                nc.n_updates = 1
                nc.D_sum = juris_D
                nc.D_sq_sum = juris_D * juris_D
                nc.D_history.append(juris_D)
                nc.context_update_counts[self.current_context_id] = 1
                self.agency_centers.append(nc)
            return

        for i, c in enumerate(same_ctx):
            kw = float(K_same[i])
            if kw < self.p.activation_threshold:
                continue
            juris_D = D * kw
            c.n_updates += 1
            c.D_sum += juris_D
            c.D_sq_sum += juris_D * juris_D
            c.context_update_counts[self.current_context_id] = \
                c.context_update_counts.get(
                    self.current_context_id, 0) + 1
            c.D_history.append(juris_D)

            if c.is_crystallized():
                dv = c.D_var_rolling()
                tw = np.clip(
                    self.p.w_max * (1.0 - dv / self.p.w_dvar_threshold),
                    -self.p.w_max, self.p.w_max)
                c.w += self.p.eta_k * kw * (tw - c.w)
                c.w = np.clip(c.w, -self.p.w_max, self.p.w_max)

            self._thermodynamic_shrink(c)

    def _crystallize_convergence(self):
        for population in [self.value_centers, self.agency_centers]:
            for c in population:
                if c.is_crystallized():
                    continue
                if c.n_updates < self.p.crystallization_threshold:
                    continue
                if len(c.D_history) < 5:
                    continue
                window = c.D_history[-50:]
                mu_D = float(np.mean(window))
                if abs(mu_D) < self.p.convergence_threshold:
                    c.mu_eff = self.p.mu_crystallized
                    c.was_ever_crystallized = True

    def _crucible_verdict(self):
        n_verified = 0
        n_dissolved = 0
        for population, use_w in [(self.value_centers, False),
                                   (self.agency_centers, True)]:
            for c in population:
                if not c.is_crystallized():
                    continue
                n_cross = len(c.D_history_cross)
                if n_cross < self.p.n_cross_min:
                    continue
                window = min(n_cross, 50)
                crucible_mu = float(np.mean(c.D_history_cross[-window:]))
                if not use_w:
                    product = c.v * crucible_mu
                    threshold = self.p.reversal_threshold
                else:
                    product = c.w * -abs(crucible_mu)
                    threshold = (self.p.reversal_threshold
                                 * (self.p.w_max / self.p.v_max))
                if product < threshold:
                    c.dissolution_log.append({
                        'step': self.current_epoch,
                        'v': float(c.v), 'w': float(c.w),
                        'mu_D_recent': float(crucible_mu),
                        'product': float(product),
                        'n_updates': c.n_updates,
                        'n_cross': n_cross,
                        'context': self.current_context_id,
                    })
                    c.mu_eff = self.p.mu_base
                    c.crucible_verified = False
                    n_dissolved += 1
                else:
                    c.crucible_verified = True
                    n_verified += 1
        return n_verified, n_dissolved

    def reset_verifications(self):
        for c in self.value_centers + self.agency_centers:
            c.crucible_verified = False
            c.D_history_cross = []

    def decay(self):
        for population in [self.value_centers, self.agency_centers]:
            for c in population:
                c.v *= (1.0 - c.mu_eff)
                c.w *= (1.0 - c.mu_eff)

    def merge_and_evict(self):
        self.value_centers = self._merge_population(self.value_centers)
        self.agency_centers = self._merge_population(self.agency_centers)

    def _merge_population(self, centers):
        if len(centers) < 2:
            return centers
        merged_indices = set()
        new_centers = []
        for i in range(len(centers)):
            if i in merged_indices:
                continue
            best = centers[i]
            for j in range(i + 1, len(centers)):
                if j in merged_indices:
                    continue
                if centers[i].context_id != centers[j].context_id:
                    continue
                dist = np.linalg.norm(centers[i].z - centers[j].z)
                ni = centers[i].sigma < self._needle_threshold
                nj = centers[j].sigma < self._needle_threshold
                if ni and nj:
                    dyn_thresh = (self._merge_ratio
                                  * max(centers[i].sigma,
                                        centers[j].sigma) * 1.5)
                else:
                    dyn_thresh = (self._merge_ratio
                                  * min(centers[i].sigma,
                                        centers[j].sigma))
                if dist < dyn_thresh:
                    other = centers[j]
                    if other.n_updates > best.n_updates:
                        best, other = other, best
                    best.v = np.clip(best.v + other.v,
                                     -self.p.v_max, self.p.v_max)
                    best.w = np.clip(best.w + other.w,
                                     -self.p.w_max, self.p.w_max)
                    best.n_updates += other.n_updates
                    best.D_sum += other.D_sum
                    best.D_sq_sum += other.D_sq_sum
                    for ctx_id, cnt in other.context_update_counts.items():
                        best.context_update_counts[ctx_id] = \
                            best.context_update_counts.get(ctx_id, 0) + cnt
                    best.D_history.extend(other.D_history)
                    best.D_history_cross.extend(other.D_history_cross)
                    best.sigma = min(best.sigma, other.sigma)
                    if other.was_ever_crystallized:
                        best.was_ever_crystallized = True
                    merged_indices.add(j)
            new_centers.append(best)
        if len(new_centers) > self.p.capacity:
            cryst = [c for c in new_centers if c.is_crystallized()]
            trans = sorted(
                [c for c in new_centers if not c.is_crystallized()],
                key=lambda c: (abs(c.v) + abs(c.w)) * c.n_updates)
            n_keep = self.p.capacity - len(cryst)
            if n_keep > 0:
                new_centers = cryst + trans[-n_keep:]
            else:
                new_centers = cryst[:self.p.capacity]
        return new_centers

    def end_epoch(self):
        self.decay()
        self._crystallize_convergence()
        n_verified, n_dissolved = self._crucible_verdict()
        self.merge_and_evict()
        D_vals = self._epoch_D_vals
        n_val = len(self.value_centers)
        n_agn = len(self.agency_centers)
        n_cryst_v = sum(1 for c in self.value_centers
                        if c.is_crystallized())
        n_cryst_a = sum(1 for c in self.agency_centers
                        if c.is_crystallized())
        n_verified_v = sum(1 for c in self.value_centers
                           if c.is_crystallized() and c.crucible_verified)
        n_verified_a = sum(1 for c in self.agency_centers
                           if c.is_crystallized() and c.crucible_verified)
        n_sat_v = sum(1 for c in self.value_centers
                      if abs(c.v) > 0.95 * self.p.v_max)
        val_sigmas = [c.sigma for c in self.value_centers]
        diag = {
            'epoch': self.current_epoch,
            'context_id': self.current_context_id,
            'n_D_updates': len(D_vals),
            'D_mean': float(np.mean(D_vals)) if D_vals else 0.0,
            'D_std': float(np.std(D_vals)) if D_vals else 0.0,
            'D_abs_mean': float(np.mean(np.abs(D_vals))) if D_vals else 0.0,
            'n_value_centers': n_val,
            'n_value_crystallized': n_cryst_v,
            'n_value_verified': n_verified_v,
            'delta_R_mean_abs': (float(np.mean([abs(c.v)
                                 for c in self.value_centers]))
                                 if self.value_centers else 0.0),
            'delta_R_max_abs': (float(max(abs(c.v)
                                for c in self.value_centers))
                                if self.value_centers else 0.0),
            'n_agency_centers': n_agn,
            'n_agency_crystallized': n_cryst_a,
            'n_agency_verified': n_verified_a,
            'delta_k_mean': (float(np.mean([c.w
                             for c in self.agency_centers]))
                             if self.agency_centers else 0.0),
            'delta_k_std': (float(np.std([c.w
                            for c in self.agency_centers]))
                            if self.agency_centers else 0.0),
            'n_centers_total': n_val + n_agn,
            'frac_crystallized': ((n_cryst_v + n_cryst_a)
                                  / max(1, n_val + n_agn)),
            'n_saturated_v': n_sat_v,
            'frac_saturated_v': n_sat_v / max(1, n_val),
            'n_verified': n_verified,
            'n_dissolved': n_dissolved,
            'sigma_mean': (float(np.mean(val_sigmas))
                           if val_sigmas else self.p.sigma),
            'sigma_min': (float(np.min(val_sigmas))
                          if val_sigmas else self.p.sigma),
            'sigma_max': (float(np.max(val_sigmas))
                          if val_sigmas else self.p.sigma),
            'n_needles': (sum(1 for s in val_sigmas
                              if s < self._needle_threshold)
                          if val_sigmas else 0),
        }
        self._epoch_D_vals = []
        self.current_epoch += 1
        return diag

    def set_context(self, context_id):
        self.current_context_id = context_id

    def snapshot(self):
        return {
            'epoch': self.current_epoch,
            'value_centers': [c.clone() for c in self.value_centers],
            'agency_centers': [c.clone() for c in self.agency_centers],
        }

    def get_stats(self):
        nv = len(self.value_centers)
        na = len(self.agency_centers)
        ncv = sum(1 for c in self.value_centers if c.is_crystallized())
        nca = sum(1 for c in self.agency_centers if c.is_crystallized())
        nvv = sum(1 for c in self.value_centers
                  if c.is_crystallized() and c.crucible_verified)
        val_sigmas = [c.sigma for c in self.value_centers]
        sig_info = ""
        if val_sigmas:
            nn_count = sum(1 for s in val_sigmas
                           if s < self._needle_threshold)
            sig_info = (f" | σ: mean={np.mean(val_sigmas):.3f}, "
                        f"needles={nn_count}")
        return (f"Value: {nv}c ({ncv}x, {nvv}v) | "
                f"Agency: {na}c ({nca}x) | "
                f"Epoch: {self.current_epoch}{sig_info}")


  CELL C6 — IBF ENGINE v4.1 (R19.6)


## CIFAR Adapter Layer

The IBF engine (C6) is identical to the chess engine — same class definitions, same lifecycle logic. The adapter layer bridges the chess-oriented API to CIFAR's pre-computed embeddings:

- `RC_encode(board)` → returns the 64D image embedding (agency space)
- `RC_encode_move(board, _, class_id)` → returns the 68D class-augmented embedding (value space)
- `RC_R_field(board)` → returns the coherence head's probability for the current class

**Critical constraint**: `R_imposed_override` must always be provided to `compute_D_and_update` in the CIFAR domain. The adapter's `RC_R_field` uses a global variable that would return stale values on the opponent-response path.

In [31]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C6b - CIFAR ADAPTER LAYER                                          ║
# ║  Bridges Chess Engine API to CIFAR-100 data.                             ║
# ║  Chess Engine calls module-level functions RC_encode, RC_encode_move,    ║
# ║  RC_R_field during compute_D_and_update. We define CIFAR-specific        ║
# ║  versions that operate on pre-computed z-vectors and a global            ║
# ║  context for the current class being evaluated.                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C6b — CIFAR ADAPTER LAYER")
print("=" * 70)

# Global adapter state:
# These globals let RC_R_field know which class is being evaluated,
# since Cell 21B's API passes only the "board" (here: image z-vector).

_ADAPTER_R_FIELD_VALUE = 0.5   # Set before each compute_D_and_update call


def RC_encode(board):
    """CIFAR adapter for Cell 21B.
    'board' is a pre-computed 64D numpy array (image embedding).
    Returns it unchanged — this is the agency-space encoding."""
    if isinstance(board, np.ndarray):
        return board
    raise TypeError(f"RC_encode expects np.ndarray, got {type(board)}")


def RC_encode_move(board_before, board_after, move):
    """CIFAR adapter for Cell 21B.
    'board_before' is the 64D image z-vector.
    'move' is the class_id (int).
    Returns the 68D class-augmented z-vector."""
    z_64 = RC_encode(board_before)
    return encode_augmented(z_64, move)


def RC_R_field(board):
    """CIFAR adapter for Cell 21B.
    Returns the pre-set coherence value for the current class.
    Must set _ADAPTER_R_FIELD_VALUE before calling compute_D_and_update.

    SAFETY NOTE: This adapter ONLY works when R_imposed_override is
    always provided to compute_D_and_update. Cell 21B also calls
    RC_R_field(board_after_opponent) when R_imposed_override is None
    and board_after_opponent is not None — that path would return a
    stale global value. The CIFAR training loop always passes
    R_imposed_override, so this path is never hit. Do NOT call
    compute_D_and_update without R_imposed_override in the CIFAR domain.
    """
    global _ADAPTER_R_FIELD_VALUE
    return _ADAPTER_R_FIELD_VALUE


def apply_move(board, move):
    """CIFAR adapter: no-op (images don't change after a 'move')."""
    return board


# ── Smoke test ────────────────────────────────────────────────────────────

print("  Adapter smoke test...")
_test_z = z64_test[0].copy()
_test_cls = test_labels_all[0]

# Test RC_encode
_z_enc = RC_encode(_test_z)
assert _z_enc.shape == (Z_DIM,), f"Expected ({Z_DIM},), got {_z_enc.shape}"

# Test RC_encode_move
_z_aug = RC_encode_move(_test_z, _test_z, _test_cls)
assert _z_aug.shape == (Z_DIM_AUG,), \
    f"Expected ({Z_DIM_AUG},), got {_z_aug.shape}"

# Test RC_R_field
_ADAPTER_R_FIELD_VALUE = 0.42
assert abs(RC_R_field(_test_z) - 0.42) < 1e-6

print("  ✓ Adapter functions verified")

# ── Build IBF engine ──────────────────────────────────────────────────────

params = IBFParams.from_calibration(C_RC)
params.sigma = SIGMA_68D
params.merge_threshold = MERGE_68D
params.sigma_agency = SIGMA_64D_AGENCY
params.v_max = 0.50
params.w_max = 5.0
params.k_0 = 5.0
params.k_min = 1.0
params.eta = 0.1
params.eta_cryst = 0.005
params.eta_k = 0.05
params.mu_base = 0.06
params.mu_crystallized = 0.001
params.crystallization_threshold = 15
params.convergence_threshold = 0.025
params.n_cross_min = 8
params.reversal_threshold = -0.0125
params.w_dvar_threshold = 0.005
params.alpha_shrink = 100.0
params.sigma_floor = 0.15
params.min_samples_shrink = 50
params.capacity = 50000

ibf = IBFEngine(params=params)

# Quick integration test: feed one CIFAR sample through the engine
ibf.set_context(0)
_ADAPTER_R_FIELD_VALUE = float(coh_probs_train[0, train_labels_all[0]])
_test_result = ibf.compute_D_and_update(
    board_before=z64_train_all[0],
    board_after_own_move=z64_train_all[0],
    board_after_opponent=None,
    move_chosen=int(train_labels_all[0]),
    R_imposed_override=1.0,
)
print(f"  Integration test: D={_test_result['D']:.6f}, "
      f"R_eff_chosen={_test_result['R_eff_chosen']:.4f}")
print(f"  Centers: val={len(ibf.value_centers)}, "
      f"agn={len(ibf.agency_centers)}")

# Reset for real training
ibf = IBFEngine(params=params)
print("  ✓ Engine ready for CIFAR training")


  CELL C6b — CIFAR ADAPTER LAYER
  Adapter smoke test...
  ✓ Adapter functions verified
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=5.0000, σ_agency=5.0000, merge=7.5000
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=8, rev_thresh=-0.0125
  Shrink: α=100.0, σ_floor=0.1500
  Integration test: D=0.000000, R_eff_chosen=0.8168
  Centers: val=1, agn=1
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=5.0000, σ_agency=5.0000, merge=7.5000
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=8, rev_thresh=-0.0125
  Shrink: α=100.0, σ_floor=0.1500
  ✓ Engine ready for CIFAR training


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# SMOKE TESTS — Run after C6b, before C7
# All three must pass before launching the full training run.
# Expected runtime: ~2-5 minutes total.
# ══════════════════════════════════════════════════════════════════════════════

import time

SMOKE_PASS = True  # Flipped to False if any test fails

# ─────────────────────────────────────────────────────────────────────────────
# SMOKE TEST 1: D signal is alive and bidirectional
# Checks: D values are non-trivial, both positive and negative.
# Failure means: coherence head is too good (no room for correction),
# too bad (random noise), or the adapter is broken.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  SMOKE TEST 1: D signal health")
print("=" * 60)

ibf_smoke = IBFEngine(params=params)
ibf_smoke.set_context(0)

t_idx_smoke = np.array(task_train_indices[0])[:50]  # 50 images only
smoke_Ds = []
smoke_Ds_correct = []
smoke_Ds_wrong = []

for i in t_idx_smoke:
    z_img = z64_train_all[i]
    true_cls = int(train_labels_all[i])
    for c in tasks[0]:
        _ADAPTER_R_FIELD_VALUE = float(coh_probs_train[i, c])
        R_truth = 0.0 if c == true_cls else 1.0  # Inverted (see C8 comment)
        r = ibf_smoke.compute_D_and_update(
            board_before=z_img, board_after_own_move=z_img,
            board_after_opponent=None, move_chosen=c,
            R_imposed_override=R_truth)
        smoke_Ds.append(r['D'])
        if c == true_cls:
            smoke_Ds_correct.append(r['D'])
        else:
            smoke_Ds_wrong.append(r['D'])

smoke_Ds = np.array(smoke_Ds)
smoke_Ds_correct = np.array(smoke_Ds_correct)
smoke_Ds_wrong = np.array(smoke_Ds_wrong)

n_pos = (smoke_Ds > 0).sum()
n_neg = (smoke_Ds < 0).sum()
abs_mean = np.abs(smoke_Ds).mean()

print(f"  Total D signals:  {len(smoke_Ds)}")
print(f"  D mean:           {smoke_Ds.mean():.4f}")
print(f"  D std:            {smoke_Ds.std():.4f}")
print(f"  |D| mean:         {abs_mean:.4f}")
print(f"  Positive:         {n_pos}  ({100*n_pos/len(smoke_Ds):.0f}%)")
print(f"  Negative:         {n_neg}  ({100*n_neg/len(smoke_Ds):.0f}%)")
print(f"  D (correct cls):  mean={smoke_Ds_correct.mean():+.4f}, "
      f"std={smoke_Ds_correct.std():.4f}")
print(f"  D (wrong cls):    mean={smoke_Ds_wrong.mean():+.4f}, "
      f"std={smoke_Ds_wrong.std():.4f}")
print(f"  Centers created:  val={len(ibf_smoke.value_centers)}, "
      f"agn={len(ibf_smoke.agency_centers)}")

# Checks
t1_ok = True
if abs_mean < 0.01:
    print("  ✗ FAIL: |D| mean < 0.01 — signal too weak")
    t1_ok = False
if n_pos == 0 or n_neg == 0:
    print("  ✗ FAIL: D is one-sided — no contrastive signal")
    t1_ok = False
if len(ibf_smoke.value_centers) == 0:
    print("  ✗ FAIL: No value centers created")
    t1_ok = False
# Polarity check: after centering, correct-class D should trend negative
# (driving v up → positive delta_R at eval), wrong-class D should trend positive
if smoke_Ds_correct.mean() > smoke_Ds_wrong.mean():
    print("  ✗ FAIL: D polarity inverted — correct class D should be "
          "lower than wrong class D")
    print(f"    (correct mean={smoke_Ds_correct.mean():+.4f}, "
          f"wrong mean={smoke_Ds_wrong.mean():+.4f})")
    t1_ok = False
else:
    print(f"  Polarity OK: correct D ({smoke_Ds_correct.mean():+.4f}) "
          f"< wrong D ({smoke_Ds_wrong.mean():+.4f})")
if t1_ok:
    print("  ✓ PASS")
else:
    SMOKE_PASS = False

del ibf_smoke


# ─────────────────────────────────────────────────────────────────────────────
# SMOKE TEST 2: Coherence head differentiates classes
# Checks: R_field values differ across task classes for the same image.
# Failure means: coherence head is undertrained or collapsed.
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
print("  SMOKE TEST 2: Coherence head class separation")
print("=" * 60)

# Check 5 images from Task 0
t2_ok = True
n_correct_top = 0
n_checked = 5

for img_i in range(n_checked):
    test_idx = task_test_indices[0][img_i]
    true_c = int(test_labels_all[test_idx])
    r_vals = {}
    for c in tasks[0]:
        r_vals[c] = float(coh_probs_test[test_idx, c])
    best_c = max(r_vals, key=r_vals.get)
    spread = max(r_vals.values()) - min(r_vals.values())
    if best_c == true_c:
        n_correct_top += 1
    print(f"  Image {img_i} (true={true_c}): "
          f"R_field={{{', '.join(f'{c}:{v:.3f}' for c, v in r_vals.items())}}}"
          f"  spread={spread:.4f}  "
          f"{'✓' if best_c == true_c else '✗'}")

# Also check global coherence head accuracy on Task 0
t0_test_idx = np.array(task_test_indices[0])
t0_true = test_labels_all[t0_test_idx]
t0_coh = coh_probs_test[t0_test_idx]
task_mask = np.full(N_CLASSES, -1e9)
for c in tasks[0]:
    task_mask[c] = 0.0
t0_preds = np.argmax(t0_coh + task_mask, axis=1)
coh_acc = (t0_preds == t0_true).mean()
print(f"\n  Coherence head Task-0 accuracy: {coh_acc:.3f}")

if coh_acc < 0.25:
    print("  ✗ FAIL: Coherence head below chance (0.20) — undertrained")
    t2_ok = False
elif coh_acc > 0.95:
    print("  ⚠ WARNING: Coherence head near-perfect — D signal may be weak")
    print("    (Proceed but watch D_abs_mean in training)")
else:
    print(f"  Good: imperfect but functional ({coh_acc:.1%})")

# Check spread: if all R_field values are identical, IBF has nothing to learn
spreads = []
for img_i in range(min(100, len(t0_test_idx))):
    test_idx = t0_test_idx[img_i]
    task_r = [float(coh_probs_test[test_idx, c]) for c in tasks[0]]
    spreads.append(max(task_r) - min(task_r))
mean_spread = np.mean(spreads)
print(f"  Mean R_field spread across task classes: {mean_spread:.4f}")

if mean_spread < 0.01:
    print("  ✗ FAIL: R_field values identical across classes — no signal")
    t2_ok = False

if t2_ok:
    print("  ✓ PASS")
else:
    SMOKE_PASS = False


# ─────────────────────────────────────────────────────────────────────────────
# SMOKE TEST 3: Runtime estimate
# Runs 1 full epoch of Task 0. Projects total training time.
# Failure means: too slow, need MAX_WRONG_CLASSES = 2.
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
print("  SMOKE TEST 3: Runtime estimate")
print("=" * 60)

ibf_rt = IBFEngine(params=params)
ibf_rt.set_context(0)

if RESET_D_CENTERING_PER_TASK:
    ibf_rt._D_running_sum = 0.0
    ibf_rt._D_running_count = 0

t_idx_rt = np.array(task_train_indices[0])
task_z_rt = z64_train_all[t_idx_rt]
task_y_rt = train_labels_all[t_idx_rt]
task_coh_rt = coh_probs_train[t_idx_rt]
perm_rt = np.random.permutation(len(task_z_rt))

n_updates = 0
t0_rt = time.time()

for local_i in perm_rt:
    z_img = task_z_rt[local_i]
    true_cls = int(task_y_rt[local_i])

    wrong_classes = [c for c in tasks[0] if c != true_cls]
    if (MAX_WRONG_CLASSES is not None
            and len(wrong_classes) > MAX_WRONG_CLASSES):
        wrong_classes = list(np.random.choice(
            wrong_classes, MAX_WRONG_CLASSES, replace=False))
    classes_to_process = [true_cls] + wrong_classes

    for c in classes_to_process:
        _ADAPTER_R_FIELD_VALUE = float(task_coh_rt[local_i, c])
        R_truth = 0.0 if c == true_cls else 1.0  # Inverted
        ibf_rt.compute_D_and_update(
            board_before=z_img, board_after_own_move=z_img,
            board_after_opponent=None, move_chosen=c,
            R_imposed_override=R_truth)
        n_updates += 1

diag_rt = ibf_rt.end_epoch()
dt_epoch1 = time.time() - t0_rt
n_centers_ep1 = diag_rt['n_value_centers']

# Later epochs are slower because kernel_batch scans more centers.
# Rough model: cost ~ N_centers, centers grow ~linearly per task.
# By task 10, expect ~5-10x more centers than epoch 1.
# Conservative multiplier: average epoch takes ~3x epoch 1.
avg_multiplier = 3.0
projected_hours = (dt_epoch1 * avg_multiplier
                   * EPOCHS_PER_TASK * N_TASKS) / 3600

print(f"  Epoch 1 time:       {dt_epoch1:.1f}s")
print(f"  D updates:          {n_updates}")
print(f"  Centers after ep1:  val={n_centers_ep1}, "
      f"agn={diag_rt['n_agency_centers']}")
print(f"  |D| mean:           {diag_rt['D_abs_mean']:.4f}")
print(f"  Images/sec:         {len(task_z_rt)/dt_epoch1:.0f}")
print(f"  Projected total:    ~{projected_hours:.1f} hours "
      f"({avg_multiplier:.0f}× slowdown factor)")

t3_ok = True
if projected_hours > 48:
    print(f"  ✗ FAIL: Projected {projected_hours:.0f}h > 48h limit")
    print(f"    → Set MAX_WRONG_CLASSES = 2 and re-run this test")
    t3_ok = False
elif projected_hours > 24:
    print(f"  ⚠ WARNING: Projected {projected_hours:.0f}h — tight")
    print(f"    Consider MAX_WRONG_CLASSES = 2 for safety margin")
else:
    print(f"  Good: {projected_hours:.1f}h is feasible")

if t3_ok:
    print("  ✓ PASS")
else:
    SMOKE_PASS = False

del ibf_rt


# ─────────────────────────────────────────────────────────────────────────────
# VERDICT
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'═' * 60}")
if SMOKE_PASS:
    print("  ALL SMOKE TESTS PASSED — proceed to C7 + C8")
else:
    print("  ✗ SMOKE TESTS FAILED — fix issues above before launching")
print(f"{'═' * 60}")

  SMOKE TEST 1: D signal health
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=10.0504, σ_agency=9.9713, merge=16.7506
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=8, rev_thresh=-0.0125
  Shrink: α=100.0, σ_floor=0.1500
  Total D signals:  250
  D mean:           0.0087
  D std:            0.2904
  |D| mean:         0.2300
  Positive:         200  (80%)
  Negative:         49  (20%)
  D (correct cls):  mean=-0.5534, std=0.1551
  D (wrong cls):    mean=+0.1492, std=0.0263
  Centers created:  val=5, agn=4
  Polarity OK: correct D (-0.5534) < wrong D (+0.1492)
  ✓ PASS

  SMOKE TEST 2: Coherence head class separation
  Image 0 (true=42): R_field={42:0.221, 41:0.005, 91:0.002, 9:0.003, 65:0.015}  spread=0.2199  ✓
  Image 1 (true=42): R_field={42:0.096, 41:0.005, 91:0.003, 9:0.003, 65:0.004}  spread=0.0931  ✓
  Image 2 (true=42): R_field={42:0.142, 41:0.005, 91:0.006, 9:0.004

## Baselines (§5.4)

All baselines operate on the same frozen 64D ViT+PCA embeddings:

- **MLP**: 64→128→128→100, vanilla SGD, no forgetting mitigation
- **Replay MLP**: same architecture + 5,000-sample reservoir buffer
- **EWC**: same architecture + Fisher information penalty (λ=1000)

The comparison tests correction dynamics on top of a shared evaluator, not end-to-end representation learning.

In [32]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C7 — BASELINES (STANDARD CLASSIFICATION)                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 70)
print("  CELL C7 — BASELINES")
print("=" * 70)


class MLPClassifier(nn.Module):
    """Standard MLP classifier: 64D → hidden → 100 logits."""
    def __init__(self, z_dim=64, hidden=128, n_cls=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_cls),
        )

    def forward(self, z):
        return self.net(z)


class ReplayClassifier(nn.Module):
    """Standard classifier + reservoir replay buffer."""
    def __init__(self, z_dim=64, hidden=128, n_cls=100,
                 buffer_size=5000):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_cls),
        )
        self.buf_z = []
        self.buf_y = []
        self.buf_size = buffer_size

    def forward(self, z):
        return self.net(z)

    def add_to_buffer(self, z_np, y_np):
        """Reservoir sampling: add batch of (z, y) numpy arrays."""
        for i in range(len(z_np)):
            if len(self.buf_z) < self.buf_size:
                self.buf_z.append(z_np[i].copy())
                self.buf_y.append(int(y_np[i]))
            else:
                j = random.randint(0, len(self.buf_z) - 1)
                self.buf_z[j] = z_np[i].copy()
                self.buf_y[j] = int(y_np[i])

    def get_replay_batch(self, batch_size=64):
        if not self.buf_z:
            return None, None
        idx = np.random.choice(len(self.buf_z),
                               min(batch_size, len(self.buf_z)),
                               replace=False)
        z = np.array([self.buf_z[i] for i in idx])
        y = np.array([self.buf_y[i] for i in idx])
        return z, y


class EWCClassifier(nn.Module):
    """Standard classifier + Elastic Weight Consolidation."""
    def __init__(self, z_dim=64, hidden=128, n_cls=100,
                 ewc_lambda=1000):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_cls),
        )
        self.ewc_lambda = ewc_lambda
        self.fisher = {}
        self.params_old = {}

    def forward(self, z):
        return self.net(z)

    def compute_fisher(self, z_np, y_np, n_samples=500):
        """Compute Fisher information from numpy arrays."""
        self.fisher = {}
        self.params_old = {}
        for name, p in self.named_parameters():
            self.fisher[name] = torch.zeros_like(p)
            self.params_old[name] = p.data.clone()

        self.train()
        n = min(n_samples, len(z_np))
        idx = np.random.choice(len(z_np), n, replace=False)
        for start in range(0, n, 64):
            batch_idx = idx[start:start + 64]
            z_b = torch.tensor(z_np[batch_idx]).float().to(DEVICE)
            y_b = torch.tensor(y_np[batch_idx]).long().to(DEVICE)
            logits = self(z_b)
            loss = nn.CrossEntropyLoss()(logits, y_b)
            self.zero_grad()
            loss.backward()
            for name, p in self.named_parameters():
                if p.grad is not None:
                    self.fisher[name] += p.grad.data ** 2
        n_batches = max(1, (n + 63) // 64)
        for name in self.fisher:
            self.fisher[name] /= n_batches

    def ewc_penalty(self):
        penalty = 0.0
        for name, p in self.named_parameters():
            if name in self.fisher:
                penalty += (self.fisher[name]
                            * (p - self.params_old[name]) ** 2).sum()
        return self.ewc_lambda * penalty


mlp_model = MLPClassifier(Z_DIM, 128, N_CLASSES).to(DEVICE)
mlp_opt = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

replay_model = ReplayClassifier(Z_DIM, 128, N_CLASSES, 5000).to(DEVICE)
replay_opt = torch.optim.Adam(replay_model.parameters(), lr=1e-3)

ewc_model = EWCClassifier(Z_DIM, 128, N_CLASSES, 1000).to(DEVICE)
ewc_opt = torch.optim.Adam(ewc_model.parameters(), lr=1e-3)

print("  MLP:    64→128→128→100 (vanilla)")
print("  Replay: 64→128→128→100 + 5000-sample buffer")
print("  EWC:    64→128→128→100 + Fisher (λ=1000)")


  CELL C7 — BASELINES
  MLP:    64→128→128→100 (vanilla)
  Replay: 64→128→128→100 + 5000-sample buffer
  EWC:    64→128→128→100 + Fisher (λ=1000)


## Stage III Training: Full IBF + Ablations + Seeds

Sequential training across 20 tasks (50 epochs each). The contrastive D-signal presents the true class with R_imposed = 0.0 (correct) and wrong classes with R_imposed = 1.0 (incorrect). The engine learns which class-augmented configurations are locally coherent.

**Runs in this cell:**
1. **Full IBF (seed 42)** — primary run with MLP/Replay/EWC baselines
2. **No-Agency (seed 42)** — tests P8 (agency should be neutral)
3. **Full IBF (seeds 123, 7)** — cross-seed replication

**What to watch for:**
- BT(log) should stay near −0.004 across all IBF conditions (P7)
- Full IBF ≈ No-Agency to three decimal places (P8)
- Baselines should catastrophically forget (MLP, EWC) or partially forget (Replay)

**Domain-specific parameters** (vs chess defaults): `alpha_shrink=10.0` (gentler than chess's 500 — CIFAR's 68D space has wider natural separation), `sigma_floor=2.5` (σ_train × 0.5, vs chess's sibling-derived 0.136).

**Important**: The paper's Table 3 reports the **log-space** Task-IL column, not the linear column.

In [34]:
import pickle

ALL_SEEDS = [42, 123, 7]
ALL_RESULTS = {}
ALL_ENGINES = {}

AGENCY_CAP = 10000


def build_task_splits(seed):
    rng = random.Random(seed)
    order = list(range(N_CLASSES))
    rng.shuffle(order)
    return [order[t * CLASSES_PER_TASK:(t + 1) * CLASSES_PER_TASK]
            for t in range(N_TASKS)]


def train_ibf(run_name, seed, agency=True, use_coherence_head=True):
    print(f"\n{'═' * 70}")
    print(f"  RUN: {run_name} (seed={seed}, agency={agency}, "
          f"coh_head={use_coherence_head})")
    print(f"{'═' * 70}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    tsks = build_task_splits(seed)
    t_train_idx = []
    t_test_idx = []
    for t in range(N_TASKS):
        tr, te = [], []
        for c in tsks[t]:
            tr.extend(train_indices_by_class[c])
            te.extend(test_indices_by_class[c])
        t_train_idx.append(tr)
        t_test_idx.append(te)

    # IBF params
    p = IBFParams.from_calibration(C_RC)
    p.sigma = SIGMA_68D
    p.merge_threshold = MERGE_68D
    p.sigma_agency = SIGMA_64D_AGENCY
    p.v_max = 0.50
    p.w_max = 5.0 if agency else 0.0
    p.k_0 = 5.0; p.k_min = 1.0
    p.eta = 0.1; p.eta_cryst = 0.005
    p.eta_k = 0.05 if agency else 0.0
    p.mu_base = 0.06; p.mu_crystallized = 0.001
    p.crystallization_threshold = 15; p.convergence_threshold = 0.025
    p.n_cross_min = 8; p.reversal_threshold = -0.0125
    p.w_dvar_threshold = 0.005; p.alpha_shrink = 10.0     # was 100, gentler shrink
    p.sigma_floor = 2.5       # was 0.15 (chess-scaled), now σ_train × 0.5
    p.min_samples_shrink = 50; p.capacity = 50000

    eng = IBFEngine(params=p)

    # Baselines only for first full-IBF seed
    train_baselines = (agency and use_coherence_head
                       and seed == ALL_SEEDS[0])
    if train_baselines:
        mlp_m = MLPClassifier(Z_DIM, 128, N_CLASSES).to(DEVICE)
        mlp_o = torch.optim.Adam(mlp_m.parameters(), lr=1e-3)
        rep_m = ReplayClassifier(Z_DIM, 128, N_CLASSES, 5000).to(DEVICE)
        rep_o = torch.optim.Adam(rep_m.parameters(), lr=1e-3)
        ewc_m = EWCClassifier(Z_DIM, 128, N_CLASSES, 1000).to(DEVICE)
        ewc_o = torch.optim.Adam(ewc_m.parameters(), lr=1e-3)

    if use_coherence_head:
        c_probs_tr = coh_probs_train
        c_probs_te = coh_probs_test
    else:
        c_probs_tr = np.full_like(coh_probs_train, 0.01)
        c_probs_te = np.full_like(coh_probs_test, 0.01)

    # Accuracy matrices: both linear and log-space
    acc_ti_lin = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ci_lin = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ti_log = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ci_log = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ti_mlp = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ti_rep = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ti_ewc = np.full((N_TASKS, N_TASKS), np.nan)

    diss_total = 0
    t_start = time.time()

    for task_id in range(N_TASKS):
        task_classes = tsks[task_id]
        all_cls = []
        for t in range(task_id + 1):
            all_cls.extend(tsks[t])

        print(f"\n  ── Task {task_id}/{N_TASKS-1} — "
              f"Classes {task_classes} ({len(all_cls)} total) ──")

        eng.set_context(task_id)
        if task_id > 0:
            eng.reset_verifications()
        if RESET_D_CENTERING_PER_TASK:
            eng._D_running_sum = 0.0
            eng._D_running_count = 0

        tidx = np.array(t_train_idx[task_id])
        tz = z64_train_all[tidx]
        ty = train_labels_all[tidx]
        tc = c_probs_tr[tidx]
        nt = len(tz)

        for epoch in range(EPOCHS_PER_TASK):
            perm = np.random.permutation(nt)
            epoch_t0 = time.time()

            # ── IBF training (linear, unchanged) ──
            for bs in range(0, nt, BATCH_SIZE):
                bidx = perm[bs:bs + BATCH_SIZE]
                for li in bidx:
                    z_img = tz[li]
                    true_cls = int(ty[li])
                    img_coh = tc[li]

                    wrong_cls = [c for c in task_classes
                                 if c != true_cls]
                    if (MAX_WRONG_CLASSES is not None
                            and len(wrong_cls) > MAX_WRONG_CLASSES):
                        wrong_cls = list(np.random.choice(
                            wrong_cls, MAX_WRONG_CLASSES, replace=False))
                    cls_proc = [true_cls] + wrong_cls

                    for c in cls_proc:
                        global _ADAPTER_R_FIELD_VALUE
                        _ADAPTER_R_FIELD_VALUE = float(img_coh[c])
                        R_truth = 0.0 if c == true_cls else 1.0
                        eng.compute_D_and_update(
                            board_before=z_img,
                            board_after_own_move=z_img,
                            board_after_opponent=None,
                            move_chosen=c,
                            R_imposed_override=R_truth)

            # ── Baseline training ──
            if train_baselines:
                for bs in range(0, nt, 64):
                    bidx = perm[bs:bs + 64]
                    z_b = torch.tensor(tz[bidx]).float().to(DEVICE)
                    y_b = torch.tensor(ty[bidx]).long().to(DEVICE)

                    mlp_m.train()
                    loss = nn.CrossEntropyLoss()(mlp_m(z_b), y_b)
                    mlp_o.zero_grad(); loss.backward(); mlp_o.step()

                    rep_m.train()
                    loss_r = nn.CrossEntropyLoss()(rep_m(z_b), y_b)
                    rz, ry = rep_m.get_replay_batch(64)
                    if rz is not None:
                        rz_t = torch.tensor(rz).float().to(DEVICE)
                        ry_t = torch.tensor(ry).long().to(DEVICE)
                        loss_r = loss_r + nn.CrossEntropyLoss()(
                            rep_m(rz_t), ry_t)
                    rep_o.zero_grad(); loss_r.backward(); rep_o.step()

                    ewc_m.train()
                    loss_e = (nn.CrossEntropyLoss()(ewc_m(z_b), y_b)
                              + ewc_m.ewc_penalty())
                    ewc_o.zero_grad(); loss_e.backward(); ewc_o.step()

                if epoch == EPOCHS_PER_TASK - 1:
                    rep_m.add_to_buffer(tz, ty)

            # ── End epoch ──
            diag = eng.end_epoch()
            diss_total += diag['n_dissolved']

            # D_history trim
            for c in eng.value_centers + eng.agency_centers:
                if len(c.D_history) > 100:
                    c.D_history = c.D_history[-100:]

            # Agency cap
            if len(eng.agency_centers) > AGENCY_CAP:
                eng.agency_centers.sort(
                    key=lambda c: c.n_updates, reverse=True)
                eng.agency_centers = eng.agency_centers[:AGENCY_CAP]

            # ── Epoch reporting ──
            if (epoch + 1) in [1, 10, 20, 30, 40, 50]:
                elapsed = time.time() - t_start
                nvc = diag['n_value_centers']
                ncx = diag['n_value_crystallized']
                nvv = diag.get('n_value_verified', 0)
                nac = len(eng.agency_centers)
                nax = sum(1 for c in eng.agency_centers
                          if c.is_crystallized())
                ep_dt = time.time() - epoch_t0
                print(f"    Ep{epoch+1:2d}: "
                      f"val={nvc}c/{ncx}x/{nvv}v, "
                      f"agn={nac}c/{nax}x, "
                      f"|D|={diag['D_abs_mean']:.4f}, "
                      f"dR_max={diag['delta_R_max_abs']:.4f}, "
                      f"diss={diag['n_dissolved']}, "
                      f"σ_min={diag['sigma_min']:.2f}, "
                      f"ep={ep_dt:.1f}s [{elapsed:.0f}s]")

        # EWC Fisher
        if train_baselines:
            ewc_m.compute_fisher(tz, ty)

        # ── Evaluate all tasks (BOTH linear and log-space) ──
        for et in range(task_id + 1):
            ec = tsks[et]
            eidx = np.array(t_test_idx[et])
            ez = z64_test[eidx]
            ey = test_labels_all[eidx]
            ecoh = c_probs_te[eidx]

            eng.set_context(et)
            cor_ti_lin, cor_ci_lin = 0, 0
            cor_ti_log, cor_ci_log = 0, 0

            for i in range(len(ez)):
                z_img = ez[i]
                tc_id = int(ey[i])

                # Task-IL
                best_lin, best_log = -1, -1
                best_r_lin, best_r_log = -1e9, -1e9
                for c in ec:
                    z_aug = encode_augmented(z_img, c)
                    dr = eng.delta_R(z_aug)
                    coh_val = float(ecoh[i, c])
                    r_lin = coh_val + dr
                    r_log = np.log(coh_val + 1e-8) + dr
                    if r_lin > best_r_lin:
                        best_r_lin = r_lin; best_lin = c
                    if r_log > best_r_log:
                        best_r_log = r_log; best_log = c
                if best_lin == tc_id: cor_ti_lin += 1
                if best_log == tc_id: cor_ti_log += 1

                # Class-IL
                best_lin_a, best_log_a = -1, -1
                best_r_lin_a, best_r_log_a = -1e9, -1e9
                for c in all_cls:
                    z_aug = encode_augmented(z_img, c)
                    dr = eng.delta_R(z_aug)
                    coh_val = float(ecoh[i, c])
                    r_lin = coh_val + dr
                    r_log = np.log(coh_val + 1e-8) + dr
                    if r_lin > best_r_lin_a:
                        best_r_lin_a = r_lin; best_lin_a = c
                    if r_log > best_r_log_a:
                        best_r_log_a = r_log; best_log_a = c
                if best_lin_a == tc_id: cor_ci_lin += 1
                if best_log_a == tc_id: cor_ci_log += 1

            n_ev = len(ez)
            acc_ti_lin[et, task_id] = cor_ti_lin / n_ev
            acc_ci_lin[et, task_id] = cor_ci_lin / n_ev
            acc_ti_log[et, task_id] = cor_ti_log / n_ev
            acc_ci_log[et, task_id] = cor_ci_log / n_ev

            # Baselines
            if train_baselines:
                with torch.no_grad():
                    z_t = torch.tensor(ez).float().to(DEVICE)
                    y_t = torch.tensor(ey).long().to(DEVICE)
                    tmask = torch.full((N_CLASSES,),
                                       float('-inf')).to(DEVICE)
                    for c in ec:
                        tmask[c] = 0.0
                    for nm, md, am in [
                        ('MLP', mlp_m, acc_ti_mlp),
                        ('Replay', rep_m, acc_ti_rep),
                        ('EWC', ewc_m, acc_ti_ewc),
                    ]:
                        md.eval()
                        preds = (md(z_t)
                                 + tmask.unsqueeze(0)).argmax(1)
                        am[et, task_id] = (
                            preds == y_t).float().mean().item()

        eng.set_context(task_id)

        # ── Task summary ──
        bt_lin, bt_log = [], []
        for t in range(task_id):
            i_lin = acc_ti_lin[t, t]
            f_lin = acc_ti_lin[t, task_id]
            i_log = acc_ti_log[t, t]
            f_log = acc_ti_log[t, task_id]
            if not (np.isnan(i_lin) or np.isnan(f_lin)):
                bt_lin.append(f_lin - i_lin)
            if not (np.isnan(i_log) or np.isnan(f_log)):
                bt_log.append(f_log - i_log)
        bt_lin_m = np.mean(bt_lin) if bt_lin else 0.0
        bt_log_m = np.mean(bt_log) if bt_log else 0.0
        cur_lin = acc_ti_lin[task_id, task_id]
        cur_log = acc_ti_log[task_id, task_id]
        print(f"  → T{task_id} done: "
              f"acc_lin={cur_lin:.3f} acc_log={cur_log:.3f}, "
              f"BT_lin={bt_lin_m:+.4f} BT_log={bt_log_m:+.4f}, "
              f"val={len(eng.value_centers)}c, "
              f"agn={len(eng.agency_centers)}c")

    # ── Compute metrics ──
    def compute_bt(m):
        bv = []
        for t in range(N_TASKS - 1):
            iv, fv = m[t, t], m[t, N_TASKS - 1]
            if not (np.isnan(iv) or np.isnan(fv)):
                bv.append(fv - iv)
        return np.mean(bv) if bv else 0.0

    def compute_avg(m):
        col = m[:, N_TASKS - 1]
        v = col[~np.isnan(col)]
        return np.mean(v) if len(v) > 0 else 0.0

    result = {
        'run_name': run_name, 'seed': seed,
        'agency': agency, 'use_coherence_head': use_coherence_head,
        'acc_taskil_lin': acc_ti_lin.tolist(),
        'acc_classil_lin': acc_ci_lin.tolist(),
        'acc_taskil_log': acc_ti_log.tolist(),
        'acc_classil_log': acc_ci_log.tolist(),
        'BT_taskil_lin': compute_bt(acc_ti_lin),
        'BT_classil_lin': compute_bt(acc_ci_lin),
        'BT_taskil_log': compute_bt(acc_ti_log),
        'BT_classil_log': compute_bt(acc_ci_log),
        'avg_taskil_lin': compute_avg(acc_ti_lin),
        'avg_classil_lin': compute_avg(acc_ci_lin),
        'avg_taskil_log': compute_avg(acc_ti_log),
        'avg_classil_log': compute_avg(acc_ci_log),
        'n_value_centers': len(eng.value_centers),
        'n_crystallized': sum(1 for c in eng.value_centers
                              if c.is_crystallized()),
        'n_verified': sum(1 for c in eng.value_centers
                          if c.is_crystallized()
                          and c.crucible_verified),
        'n_dissolved': diss_total,
        'n_agency_centers': len(eng.agency_centers),
        'elapsed': time.time() - t_start,
    }

    if train_baselines:
        result['acc_taskil_mlp'] = acc_ti_mlp.tolist()
        result['acc_taskil_replay'] = acc_ti_rep.tolist()
        result['acc_taskil_ewc'] = acc_ti_ewc.tolist()
        result['BT_mlp'] = compute_bt(acc_ti_mlp)
        result['BT_replay'] = compute_bt(acc_ti_rep)
        result['BT_ewc'] = compute_bt(acc_ti_ewc)
        result['avg_mlp'] = compute_avg(acc_ti_mlp)
        result['avg_replay'] = compute_avg(acc_ti_rep)
        result['avg_ewc'] = compute_avg(acc_ti_ewc)

    # ── Save engine state BEFORE σ sweep ──
    engine_state = {
        'value_centers': [(c.z.tolist(), float(c.v), float(c.w),
                           float(c.sigma), int(c.context_id),
                           bool(c.is_crystallized()),
                           bool(c.crucible_verified),
                           int(c.n_updates),
                           float(c.D_var_rolling()))
                          for c in eng.value_centers],
        'n_agency': len(eng.agency_centers),
        'params': {'sigma': eng.p.sigma,
                   'merge': eng.p.merge_threshold,
                   'sigma_agency': eng._sigma_agency},
    }
    pkl_name = f"overnight_{run_name.replace(' ', '_')}_engine.pkl"
    with open(os.path.join(OUT_DIR, pkl_name), 'wb') as f:
        pickle.dump(engine_state, f)
    print(f"  Engine saved: {pkl_name}")

    # ── σ_eval sweep (finer grid around 1.0) ──
    orig_sigmas = [c.sigma for c in eng.value_centers]
    sweep_lin, sweep_log = {}, {}
    for scale in [0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.3, 1.5]:
        s_eval = SIGMA_68D * scale
        for c in eng.value_centers:
            c.sigma = s_eval
        accs_lin, accs_log = [], []
        for t in random.sample(range(N_TASKS), min(5, N_TASKS)):
            ec = tsks[t]
            eidx = np.array(t_test_idx[t])
            ez = z64_test[eidx]
            ey = test_labels_all[eidx]
            ecoh = c_probs_te[eidx]
            eng.set_context(t)
            cor_lin, cor_log = 0, 0
            for i in range(len(ez)):
                best_lin, best_log = -1, -1
                best_r_lin, best_r_log = -1e9, -1e9
                for c in ec:
                    z_aug = encode_augmented(ez[i], c)
                    dr = eng.delta_R(z_aug)
                    cv = float(ecoh[i, c])
                    r_lin = cv + dr
                    r_log = np.log(cv + 1e-8) + dr
                    if r_lin > best_r_lin:
                        best_r_lin = r_lin; best_lin = c
                    if r_log > best_r_log:
                        best_r_log = r_log; best_log = c
                if best_lin == int(ey[i]): cor_lin += 1
                if best_log == int(ey[i]): cor_log += 1
            accs_lin.append(cor_lin / len(ez))
            accs_log.append(cor_log / len(ez))
        sweep_lin[scale] = float(np.mean(accs_lin))
        sweep_log[scale] = float(np.mean(accs_log))

    for c, s in zip(eng.value_centers, orig_sigmas):
        c.sigma = s
    result['sigma_sweep_lin'] = sweep_lin
    result['sigma_sweep_log'] = sweep_log

    # Save results JSON
    cp_name = f"overnight_{run_name.replace(' ', '_')}.json"
    with open(os.path.join(OUT_DIR, cp_name), 'w') as f:
        json.dump(result, f, indent=2)

    print(f"\n  {run_name} DONE:")
    print(f"    Linear:  avg={result['avg_taskil_lin']:.4f}, "
          f"BT={result['BT_taskil_lin']:+.4f}")
    print(f"    Log:     avg={result['avg_taskil_log']:.4f}, "
          f"BT={result['BT_taskil_log']:+.4f}")
    print(f"    Centers: val={len(eng.value_centers)}, "
          f"agn={len(eng.agency_centers)}, diss={diss_total}")
    print(f"    Time:    {result['elapsed']/3600:.1f}h")
    print(f"    σ_sweep lin: {sweep_lin}")
    print(f"    σ_sweep log: {sweep_log}")

    ALL_ENGINES[run_name] = eng
    return result


# ── Coherence head baseline ──────────────────────────────────────────

print("Coherence head baseline (Task-IL, all tasks, seed 42):")
tsks_42 = build_task_splits(42)
coh_baseline_accs = []
for t in range(N_TASKS):
    ec = tsks_42[t]
    eidx = []
    for c in ec:
        eidx.extend(test_indices_by_class[c])
    eidx = np.array(eidx)
    ey = test_labels_all[eidx]
    ecoh = coh_probs_test[eidx]
    tmask = np.full(N_CLASSES, -1e9)
    for c in ec:
        tmask[c] = 0.0
    preds = np.argmax(ecoh + tmask, axis=1)
    acc = (preds == ey).mean()
    coh_baseline_accs.append(acc)
mean_coh = np.mean(coh_baseline_accs)
print(f"  Mean: {mean_coh:.3f}, "
      f"range: [{min(coh_baseline_accs):.3f}, "
      f"{max(coh_baseline_accs):.3f}]")


# ══════════════════════════════════════════════════════════════════════
# RUN ALL
# ══════════════════════════════════════════════════════════════════════

ALL_RESULTS['full_42'] = train_ibf(
    'Full IBF s42', seed=42, agency=True, use_coherence_head=True)

ALL_RESULTS['noagn_42'] = train_ibf(
    'NoAgency s42', seed=42, agency=False, use_coherence_head=True)

ALL_RESULTS['full_123'] = train_ibf(
    'Full IBF s123', seed=123, agency=True, use_coherence_head=True)

ALL_RESULTS['full_7'] = train_ibf(
    'Full IBF s7', seed=7, agency=True, use_coherence_head=True)


# ══════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════

print(f"\n{'═' * 78}")
print(f"  FINAL RESULTS — σ={SIGMA_68D}, log-space eval")
print(f"{'═' * 78}")
print(f"  Coherence head baseline (Task-IL): {mean_coh:.3f}")

print(f"\n  {'Run':<20s}  {'Avg(log)':>9s}  {'BT(log)':>9s}  "
      f"{'Avg(lin)':>9s}  {'BT(lin)':>9s}  "
      f"{'Val':>5s}  {'Diss':>6s}  {'Agn':>5s}  {'Time':>6s}")
print(f"  {'─' * 78}")

for key in ['full_42', 'noagn_42', 'full_123', 'full_7']:
    r = ALL_RESULTS[key]
    print(f"  {r['run_name']:<20s}  "
          f"{r['avg_taskil_log']:>9.4f}  "
          f"{r['BT_taskil_log']:>+9.4f}  "
          f"{r['avg_taskil_lin']:>9.4f}  "
          f"{r['BT_taskil_lin']:>+9.4f}  "
          f"{r['n_value_centers']:>5d}  "
          f"{r['n_dissolved']:>6d}  "
          f"{r.get('n_agency_centers', 0):>5d}  "
          f"{r['elapsed']/3600:>5.1f}h")

r42 = ALL_RESULTS['full_42']
if 'BT_mlp' in r42:
    print(f"\n  Baselines (seed 42, Task-IL):")
    for bl in ['mlp', 'replay', 'ewc']:
        print(f"    {bl.upper():<10s}  "
              f"avg={r42[f'avg_{bl}']:.4f}  "
              f"BT={r42[f'BT_{bl}']:+.4f}")

# Cross-seed
full_runs = [ALL_RESULTS[k] for k in ['full_42', 'full_123', 'full_7']]
avg_log = [r['avg_taskil_log'] for r in full_runs]
bt_log = [r['BT_taskil_log'] for r in full_runs]
avg_lin = [r['avg_taskil_lin'] for r in full_runs]
bt_lin = [r['BT_taskil_lin'] for r in full_runs]
print(f"\n  Cross-seed (Full IBF, n=3):")
print(f"    Log:    avg={np.mean(avg_log):.4f}±{np.std(avg_log):.4f}, "
      f"BT={np.mean(bt_log):+.4f}±{np.std(bt_log):.4f}")
print(f"    Linear: avg={np.mean(avg_lin):.4f}±{np.std(avg_lin):.4f}, "
      f"BT={np.mean(bt_lin):+.4f}±{np.std(bt_lin):.4f}")

# Class-IL summary
avg_ci_log = [r['avg_classil_log'] for r in full_runs]
bt_ci_log = [r['BT_classil_log'] for r in full_runs]
print(f"    Class-IL log: avg={np.mean(avg_ci_log):.4f}±"
      f"{np.std(avg_ci_log):.4f}, "
      f"BT={np.mean(bt_ci_log):+.4f}±{np.std(bt_ci_log):.4f}")

# Save all
with open(os.path.join(OUT_DIR, 'overnight_all_results.json'), 'w') as f:
    json.dump(ALL_RESULTS, f, indent=2)
print(f"\n  All results saved to {OUT_DIR}/overnight_all_results.json")

print(f"\n{'═' * 78}")
print(f"  OVERNIGHT RUN COMPLETE")
print(f"{'═' * 78}")

Coherence head baseline (Task-IL, all tasks, seed 42):
  Mean: 0.901, range: [0.748, 0.978]

══════════════════════════════════════════════════════════════════════
  RUN: Full IBF s42 (seed=42, agency=True, coh_head=True)
══════════════════════════════════════════════════════════════════════
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=5.0000, σ_agency=5.0000, merge=7.5000
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=8, rev_thresh=-0.0125
  Shrink: α=10.0, σ_floor=1.2500

  ── Task 0/19 — Classes [42, 41, 91, 9, 65] (5 total) ──
    Ep 1: val=125c/106x/0v, agn=28c/28x, |D|=0.2546, dR_max=0.4700, diss=0, σ_min=4.71, ep=3.0s [3s]
    Ep10: val=153c/146x/0v, agn=61c/61x, |D|=0.2458, dR_max=0.4995, diss=0, σ_min=2.82, ep=4.9s [47s]
    Ep20: val=155c/149x/0v, agn=67c/67x, |D|=0.2455, dR_max=0.4995, diss=0, σ_min=2.30, ep=5.0s [96s]
    Ep30: val=156c/150x/0v, agn=69c/69x, 

## Additional Ablations — No-Crucible + No-Crystallization

These complete the four-condition ablation set for CIFAR (§7.3):

The paper predicts that **all four conditions cluster** around the same performance because spatial isolation alone is sufficient for retention at 68D. The higher-order mechanisms (crystallization, Crucible, agency) still operate — thousands of crystallized centers, tens of thousands of dissolutions — but their behavioral contribution is masked because inter-task distances are large relative to the calibrated bandwidth.

This is the correct null result: the mechanisms are present but redundant when geometry already prevents interference.

In [37]:
# ══════════════════════════════════════════════════════════════════════════════
# ABLATION CELL — No-Crucible + No-Crystallization (seed 42)
# Reuses train_ibf pattern with param overrides.
# All existing results in ALL_RESULTS / ALL_ENGINES are preserved.
# ══════════════════════════════════════════════════════════════════════════════

def train_ibf_ablation(run_name, seed, param_overrides=None,
                       agency=True, use_coherence_head=True,
                       c_probs_tr=None, c_probs_te=None):

    print(f"\n{'═' * 70}")
    print(f"  RUN: {run_name} (seed={seed})")
    print(f"  Overrides: {param_overrides}")
    print(f"{'═' * 70}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    tsks = build_task_splits(seed)
    t_train_idx = []
    t_test_idx = []
    for t in range(N_TASKS):
        tr, te = [], []
        for c in tsks[t]:
            tr.extend(train_indices_by_class[c])
            te.extend(test_indices_by_class[c])
        t_train_idx.append(tr)
        t_test_idx.append(te)

    p = IBFParams.from_calibration(C_RC)
    p.sigma = SIGMA_68D
    p.merge_threshold = MERGE_68D
    p.sigma_agency = SIGMA_64D_AGENCY
    p.v_max = 0.50
    p.w_max = 5.0 if agency else 0.0
    p.k_0 = 5.0; p.k_min = 1.0
    p.eta = 0.1; p.eta_cryst = 0.005
    p.eta_k = 0.05 if agency else 0.0
    p.mu_base = 0.06; p.mu_crystallized = 0.001
    p.crystallization_threshold = 15; p.convergence_threshold = 0.025
    p.n_cross_min = 8; p.reversal_threshold = -0.0125
    p.w_dvar_threshold = 0.005; p.alpha_shrink = 10.0
    p.sigma_floor = 1.25; p.min_samples_shrink = 50; p.capacity = 50000

    if param_overrides:
        for k, v in param_overrides.items():
            if hasattr(p, k):
                setattr(p, k, v)
                print(f"  Override: {k} = {v}")
            else:
                print(f"  WARNING: unknown param {k}")

    eng = IBFEngine(params=p)

    if c_probs_tr is None:
        c_probs_tr = coh_probs_train if use_coherence_head else np.full_like(coh_probs_train, 0.01)
    if c_probs_te is None:
        c_probs_te = coh_probs_test if use_coherence_head else np.full_like(coh_probs_test, 0.01)

    acc_ti_lin = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ci_lin = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ti_log = np.full((N_TASKS, N_TASKS), np.nan)
    acc_ci_log = np.full((N_TASKS, N_TASKS), np.nan)

    diss_total = 0
    t_start = time.time()

    for task_id in range(N_TASKS):
        task_classes = tsks[task_id]
        all_cls = []
        for t in range(task_id + 1):
            all_cls.extend(tsks[t])

        print(f"\n  ── Task {task_id}/{N_TASKS-1} — "
              f"Classes {task_classes} ({len(all_cls)} total) ──")

        eng.set_context(task_id)
        if task_id > 0:
            eng.reset_verifications()
        if RESET_D_CENTERING_PER_TASK:
            eng._D_running_sum = 0.0
            eng._D_running_count = 0

        tidx = np.array(t_train_idx[task_id])
        tz = z64_train_all[tidx]
        ty = train_labels_all[tidx]
        tc = c_probs_tr[tidx]
        nt = len(tz)

        for epoch in range(EPOCHS_PER_TASK):
            perm = np.random.permutation(nt)
            epoch_t0 = time.time()

            for bs in range(0, nt, BATCH_SIZE):
                bidx = perm[bs:bs + BATCH_SIZE]
                for li in bidx:
                    z_img = tz[li]
                    true_cls = int(ty[li])
                    img_coh = tc[li]

                    wrong_cls = [c for c in task_classes if c != true_cls]
                    if (MAX_WRONG_CLASSES is not None
                            and len(wrong_cls) > MAX_WRONG_CLASSES):
                        wrong_cls = list(np.random.choice(
                            wrong_cls, MAX_WRONG_CLASSES, replace=False))
                    cls_proc = [true_cls] + wrong_cls

                    for c in cls_proc:
                        global _ADAPTER_R_FIELD_VALUE
                        _ADAPTER_R_FIELD_VALUE = float(img_coh[c])
                        R_truth = 0.0 if c == true_cls else 1.0
                        eng.compute_D_and_update(
                            board_before=z_img,
                            board_after_own_move=z_img,
                            board_after_opponent=None,
                            move_chosen=c,
                            R_imposed_override=R_truth)

            diag = eng.end_epoch()
            diss_total += diag['n_dissolved']

            for c in eng.value_centers + eng.agency_centers:
                if len(c.D_history) > 100:
                    c.D_history = c.D_history[-100:]

            if len(eng.agency_centers) > AGENCY_CAP:
                eng.agency_centers.sort(
                    key=lambda c: c.n_updates, reverse=True)
                eng.agency_centers = eng.agency_centers[:AGENCY_CAP]

            if (epoch + 1) in [1, 10, 20, 30, 40, 50]:
                elapsed = time.time() - t_start
                nvc = diag['n_value_centers']
                ncx = diag['n_value_crystallized']
                nvv = diag.get('n_value_verified', 0)
                nac = len(eng.agency_centers)
                nax = sum(1 for c in eng.agency_centers
                          if c.is_crystallized())
                ep_dt = time.time() - epoch_t0
                print(f"    Ep{epoch+1:2d}: "
                      f"val={nvc}c/{ncx}x/{nvv}v, "
                      f"agn={nac}c/{nax}x, "
                      f"|D|={diag['D_abs_mean']:.4f}, "
                      f"dR_max={diag['delta_R_max_abs']:.4f}, "
                      f"diss={diag['n_dissolved']}, "
                      f"σ_min={diag['sigma_min']:.2f}, "
                      f"ep={ep_dt:.1f}s [{elapsed:.0f}s]")

        # Evaluate all tasks
        for et in range(task_id + 1):
            ec = tsks[et]
            eidx = np.array(t_test_idx[et])
            ez = z64_test[eidx]
            ey = test_labels_all[eidx]
            ecoh = c_probs_te[eidx]

            eng.set_context(et)
            cor_ti_lin, cor_ci_lin = 0, 0
            cor_ti_log, cor_ci_log = 0, 0

            for i in range(len(ez)):
                z_img = ez[i]
                tc_id = int(ey[i])

                best_lin, best_log = -1, -1
                best_r_lin, best_r_log = -1e9, -1e9
                for c in ec:
                    z_aug = encode_augmented(z_img, c)
                    dr = eng.delta_R(z_aug)
                    coh_val = float(ecoh[i, c])
                    r_lin = coh_val + dr
                    r_log = np.log(coh_val + 1e-8) + dr
                    if r_lin > best_r_lin:
                        best_r_lin = r_lin; best_lin = c
                    if r_log > best_r_log:
                        best_r_log = r_log; best_log = c
                if best_lin == tc_id: cor_ti_lin += 1
                if best_log == tc_id: cor_ti_log += 1

                best_lin_a, best_log_a = -1, -1
                best_r_lin_a, best_r_log_a = -1e9, -1e9
                for c in all_cls:
                    z_aug = encode_augmented(z_img, c)
                    dr = eng.delta_R(z_aug)
                    coh_val = float(ecoh[i, c])
                    r_lin = coh_val + dr
                    r_log = np.log(coh_val + 1e-8) + dr
                    if r_lin > best_r_lin_a:
                        best_r_lin_a = r_lin; best_lin_a = c
                    if r_log > best_r_log_a:
                        best_r_log_a = r_log; best_log_a = c
                if best_lin_a == tc_id: cor_ci_lin += 1
                if best_log_a == tc_id: cor_ci_log += 1

            n_ev = len(ez)
            acc_ti_lin[et, task_id] = cor_ti_lin / n_ev
            acc_ci_lin[et, task_id] = cor_ci_lin / n_ev
            acc_ti_log[et, task_id] = cor_ti_log / n_ev
            acc_ci_log[et, task_id] = cor_ci_log / n_ev

        eng.set_context(task_id)

        # Task summary
        bt_lin, bt_log = [], []
        for t in range(task_id):
            i_lin = acc_ti_lin[t, t]
            f_lin = acc_ti_lin[t, task_id]
            i_log = acc_ti_log[t, t]
            f_log = acc_ti_log[t, task_id]
            if not (np.isnan(i_lin) or np.isnan(f_lin)):
                bt_lin.append(f_lin - i_lin)
            if not (np.isnan(i_log) or np.isnan(f_log)):
                bt_log.append(f_log - i_log)
        bt_lin_m = np.mean(bt_lin) if bt_lin else 0.0
        bt_log_m = np.mean(bt_log) if bt_log else 0.0
        cur_lin = acc_ti_lin[task_id, task_id]
        cur_log = acc_ti_log[task_id, task_id]
        print(f"  → T{task_id} done: "
              f"acc_lin={cur_lin:.3f} acc_log={cur_log:.3f}, "
              f"BT_lin={bt_lin_m:+.4f} BT_log={bt_log_m:+.4f}, "
              f"val={len(eng.value_centers)}c, "
              f"agn={len(eng.agency_centers)}c")

    # Final metrics
    def compute_bt(m):
        bv = []
        for t in range(N_TASKS - 1):
            iv, fv = m[t, t], m[t, N_TASKS - 1]
            if not (np.isnan(iv) or np.isnan(fv)):
                bv.append(fv - iv)
        return np.mean(bv) if bv else 0.0

    def compute_avg(m):
        col = m[:, N_TASKS - 1]
        v = col[~np.isnan(col)]
        return np.mean(v) if len(v) > 0 else 0.0

    result = {
        'run_name': run_name, 'seed': seed,
        'param_overrides': str(param_overrides),
        'acc_taskil_lin': acc_ti_lin.tolist(),
        'acc_classil_lin': acc_ci_lin.tolist(),
        'acc_taskil_log': acc_ti_log.tolist(),
        'acc_classil_log': acc_ci_log.tolist(),
        'BT_taskil_lin': compute_bt(acc_ti_lin),
        'BT_classil_lin': compute_bt(acc_ci_lin),
        'BT_taskil_log': compute_bt(acc_ti_log),
        'BT_classil_log': compute_bt(acc_ci_log),
        'avg_taskil_lin': compute_avg(acc_ti_lin),
        'avg_classil_lin': compute_avg(acc_ci_lin),
        'avg_taskil_log': compute_avg(acc_ti_log),
        'avg_classil_log': compute_avg(acc_ci_log),
        'n_value_centers': len(eng.value_centers),
        'n_crystallized': sum(1 for c in eng.value_centers
                              if c.is_crystallized()),
        'n_verified': sum(1 for c in eng.value_centers
                          if c.is_crystallized() and c.crucible_verified),
        'n_dissolved': diss_total,
        'n_agency_centers': len(eng.agency_centers),
        'elapsed': time.time() - t_start,
    }

    # Save engine state
    engine_state = {
        'value_centers': [(c.z.tolist(), float(c.v), float(c.w),
                           float(c.sigma), int(c.context_id),
                           bool(c.is_crystallized()),
                           bool(c.crucible_verified),
                           int(c.n_updates),
                           float(c.D_var_rolling()))
                          for c in eng.value_centers],
        'n_agency': len(eng.agency_centers),
        'params': {'sigma': eng.p.sigma,
                   'merge': eng.p.merge_threshold,
                   'sigma_agency': eng._sigma_agency},
    }
    pkl_name = f"ablation_{run_name.replace(' ', '_')}_engine.pkl"
    with open(os.path.join(OUT_DIR, pkl_name), 'wb') as f:
        pickle.dump(engine_state, f)

    cp_name = f"ablation_{run_name.replace(' ', '_')}.json"
    with open(os.path.join(OUT_DIR, cp_name), 'w') as f:
        json.dump(result, f, indent=2)

    print(f"\n  {run_name} DONE:")
    print(f"    Log:  avg={result['avg_taskil_log']:.4f}, "
          f"BT={result['BT_taskil_log']:+.4f}")
    print(f"    Lin:  avg={result['avg_taskil_lin']:.4f}, "
          f"BT={result['BT_taskil_lin']:+.4f}")
    print(f"    Centers: val={len(eng.value_centers)}, "
          f"cryst={result['n_crystallized']}, "
          f"verified={result['n_verified']}, "
          f"diss={diss_total}")
    print(f"    Time: {result['elapsed']/3600:.1f}h")

    ALL_ENGINES[run_name] = eng
    ALL_RESULTS[run_name] = result
    return result


# ══════════════════════════════════════════════════════════════════════
# RUN ABLATIONS
# ══════════════════════════════════════════════════════════════════════

r_nocruc = train_ibf_ablation(
    'NoCrucible s42', seed=42,
    param_overrides={'n_cross_min': 999999})

r_nocryst = train_ibf_ablation(
    'NoCryst s42', seed=42,
    param_overrides={'crystallization_threshold': 999999})


# ══════════════════════════════════════════════════════════════════════
# COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print(f"  ABLATION COMPARISON (seed 42, Task-IL)")
print(f"{'═' * 70}")
print(f"  Coherence head baseline: 0.901")
print(f"\n  {'Run':<22s}  {'Avg(log)':>9s}  {'BT(log)':>9s}  "
      f"{'Val':>5s}  {'Cryst':>6s}  {'Verif':>6s}  {'Diss':>6s}")
print(f"  {'─' * 70}")

ref = ALL_RESULTS['full_42']
print(f"  {'Full IBF s42':<22s}  "
      f"{ref['avg_taskil_log']:>9.4f}  {ref['BT_taskil_log']:>+9.4f}  "
      f"{ref['n_value_centers']:>5d}  "
      f"{ref['n_crystallized']:>6d}  "
      f"{ref['n_verified']:>6d}  "
      f"{ref['n_dissolved']:>6d}")

for key in ['NoCrucible s42', 'NoCryst s42']:
    r = ALL_RESULTS[key]
    print(f"  {r['run_name']:<22s}  "
          f"{r['avg_taskil_log']:>9.4f}  {r['BT_taskil_log']:>+9.4f}  "
          f"{r['n_value_centers']:>5d}  "
          f"{r['n_crystallized']:>6d}  "
          f"{r['n_verified']:>6d}  "
          f"{r['n_dissolved']:>6d}")

ref_na = ALL_RESULTS['noagn_42']
print(f"  {'NoAgency s42':<22s}  "
      f"{ref_na['avg_taskil_log']:>9.4f}  {ref_na['BT_taskil_log']:>+9.4f}  "
      f"{ref_na['n_value_centers']:>5d}  "
      f"{ref_na['n_crystallized']:>6d}  "
      f"{ref_na['n_verified']:>6d}  "
      f"{ref_na['n_dissolved']:>6d}")

print(f"\n  Baselines:")
print(f"    Replay:  avg=0.7226, BT=-0.2340")
print(f"    MLP:     avg=0.2761, BT=-0.7187")
print(f"    EWC:     avg=0.3114, BT=-0.6811")
print(f"{'═' * 70}")


══════════════════════════════════════════════════════════════════════
  RUN: NoCrucible s42 (seed=42)
  Overrides: {'n_cross_min': 999999}
══════════════════════════════════════════════════════════════════════
  Override: n_cross_min = 999999
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=5.0000, σ_agency=5.0000, merge=7.5000
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=999999, rev_thresh=-0.0125
  Shrink: α=10.0, σ_floor=1.2500

  ── Task 0/19 — Classes [42, 41, 91, 9, 65] (5 total) ──
    Ep 1: val=125c/106x/0v, agn=28c/28x, |D|=0.2546, dR_max=0.4700, diss=0, σ_min=4.71, ep=3.0s [3s]
    Ep10: val=153c/146x/0v, agn=61c/61x, |D|=0.2458, dR_max=0.4995, diss=0, σ_min=2.82, ep=4.9s [47s]
    Ep20: val=155c/149x/0v, agn=67c/67x, |D|=0.2455, dR_max=0.4995, diss=0, σ_min=2.30, ep=5.3s [98s]
    Ep30: val=156c/150x/0v, agn=69c/69x, |D|=0.2453, dR_max=0.4995, diss=0, σ_min=2.

## Weak-Head Experiment — Corrections Are Real (§7.3)

The strong coherence head (10/class, 90.1% baseline) leaves very little room for gain. To test whether IBF corrections carry genuine signal, we repeat the experiment with a deliberately weakened head trained on only 2 samples per class (~72.6% baseline).

The paper reports: *"Under that weaker prior, IBF rises from 0.726 to 0.739 (+1.3%), improves 16 of 20 tasks, and reaches a maximum single-task gain of +8.2%."*

This confirms that the corrections are real — they are simply operating against a strong ceiling in the standard configuration.

In [38]:
# ══════════════════════════════════════════════════════════════════════════════
# WEAKENED COHERENCE HEAD (2 images/class → ~60-70% baseline)
# Tests whether IBF corrections add value when R_field is genuinely imperfect.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  TRAINING WEAKENED COHERENCE HEAD (2/class)")
print("=" * 70)

# ── Train weak head ──
weak_head = CoherenceHead(Z_DIM, N_CLASSES).to(DEVICE)
weak_opt = torch.optim.Adam(weak_head.parameters(), lr=1e-3)

weak_idx = []
for cls in range(N_CLASSES):
    cls_idx = train_indices_by_class[cls][:2]  # 2 per class
    weak_idx.extend(cls_idx)
random.shuffle(weak_idx)

weak_z = z64_train_all[weak_idx]
weak_y = train_labels_all[weak_idx]
print(f"  Training on {len(weak_z)} samples (2/class)")

for epoch in range(10):
    weak_head.train()
    perm = np.random.permutation(len(weak_z))
    correct, total = 0, 0
    for start in range(0, len(weak_z), 64):
        bidx = perm[start:start + 64]
        z_b = torch.tensor(weak_z[bidx]).float().to(DEVICE)
        y_b = torch.tensor(weak_y[bidx]).long().to(DEVICE)
        probs = weak_head(z_b)
        loss = nn.CrossEntropyLoss()(torch.log(probs + 1e-10), y_b)
        weak_opt.zero_grad()
        loss.backward()
        weak_opt.step()
        correct += (probs.argmax(1) == y_b).sum().item()
        total += len(y_b)
    if (epoch + 1) % 5 == 0:
        print(f"    Epoch {epoch+1}/10: acc={correct/total:.3f}")

for param in weak_head.parameters():
    param.requires_grad = False
weak_head.eval()
del weak_opt

# ── Pre-compute weak coherence probs ──
print("  Pre-computing weak coherence probabilities...")
with torch.no_grad():
    weak_probs_test = []
    for start in range(0, len(z64_test), 256):
        z_b = torch.tensor(z64_test[start:start+256]).float().to(DEVICE)
        probs = weak_head(z_b).cpu().numpy()
        weak_probs_test.append(probs)
    weak_probs_test = np.concatenate(weak_probs_test, axis=0)

    weak_probs_train = []
    for start in range(0, len(z64_train_all), 256):
        z_b = torch.tensor(z64_train_all[start:start+256]).float().to(DEVICE)
        probs = weak_head(z_b).cpu().numpy()
        weak_probs_train.append(probs)
    weak_probs_train = np.concatenate(weak_probs_train, axis=0)

# ── Measure weak baseline ──
print("\n  Weak coherence head baseline (Task-IL):")
tsks_42 = build_task_splits(42)
weak_baseline_accs = []
for t in range(N_TASKS):
    ec = tsks_42[t]
    eidx = []
    for c in ec:
        eidx.extend(test_indices_by_class[c])
    eidx = np.array(eidx)
    ey = test_labels_all[eidx]
    ecoh = weak_probs_test[eidx]
    tmask = np.full(N_CLASSES, -1e9)
    for c in ec:
        tmask[c] = 0.0
    preds = np.argmax(ecoh + tmask, axis=1)
    acc = (preds == ey).mean()
    weak_baseline_accs.append(acc)
weak_mean = np.mean(weak_baseline_accs)
print(f"  Mean: {weak_mean:.3f}, "
      f"range: [{min(weak_baseline_accs):.3f}, "
      f"{max(weak_baseline_accs):.3f}]")
if 0.50 < weak_mean < 0.80:
    print(f"  ✓ In target range")
else:
    print(f"  ⚠ Outside target (0.50-0.80)")


# ══════════════════════════════════════════════════════════════════════
# RUN IBF WITH WEAK HEAD
# ══════════════════════════════════════════════════════════════════════

r_weak = train_ibf_ablation(
    'WeakHead s42', seed=42,
    c_probs_tr=weak_probs_train,
    c_probs_te=weak_probs_test)


# ══════════════════════════════════════════════════════════════════════
# COMPARISON
# ══════════════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print(f"  WEAK HEAD COMPARISON (seed 42, Task-IL)")
print(f"{'═' * 70}")
print(f"  Strong head baseline (10/class): {0.901:.3f}")
print(f"  Weak head baseline (2/class):    {weak_mean:.3f}")
print(f"  Gap available for correction:    {1.0 - weak_mean:.3f}")

r_w = ALL_RESULTS['WeakHead s42']
r_f = ALL_RESULTS['full_42']

print(f"\n  {'Condition':<25s}  {'Baseline':>9s}  {'IBF(log)':>9s}  "
      f"{'Δ vs base':>9s}  {'BT(log)':>9s}")
print(f"  {'─' * 65}")
print(f"  {'Strong head (10/class)':<25s}  {'0.901':>9s}  "
      f"{r_f['avg_taskil_log']:>9.4f}  "
      f"{r_f['avg_taskil_log'] - 0.901:>+9.4f}  "
      f"{r_f['BT_taskil_log']:>+9.4f}")
print(f"  {'Weak head (2/class)':<25s}  {weak_mean:>9.3f}  "
      f"{r_w['avg_taskil_log']:>9.4f}  "
      f"{r_w['avg_taskil_log'] - weak_mean:>+9.4f}  "
      f"{r_w['BT_taskil_log']:>+9.4f}")
print(f"{'═' * 70}")

  TRAINING WEAKENED COHERENCE HEAD (2/class)
  Training on 200 samples (2/class)
    Epoch 5/10: acc=0.265
    Epoch 10/10: acc=0.695
  Pre-computing weak coherence probabilities...

  Weak coherence head baseline (Task-IL):
  Mean: 0.726, range: [0.586, 0.862]
  ✓ In target range

══════════════════════════════════════════════════════════════════════
  RUN: WeakHead s42 (seed=42)
  Overrides: None
══════════════════════════════════════════════════════════════════════
IBF Engine v4.1 (R19.6) initialized
  η=0.1, η_cryst=0.005, μ_base=0.06
  η/μ transient=1.67, crystal=5.0
  k_0=5.0, v_max=0.5, w_max=5.0
  σ_value=5.0000, σ_agency=5.0000, merge=7.5000
  Convergence: |μ_D|<0.025, n_min=15
  Crucible: n_cross_min=8, rev_thresh=-0.0125
  Shrink: α=10.0, σ_floor=1.2500

  ── Task 0/19 — Classes [42, 41, 91, 9, 65] (5 total) ──
    Ep 1: val=130c/107x/0v, agn=31c/31x, |D|=0.2972, dR_max=0.4700, diss=0, σ_min=4.56, ep=3.1s [3s]
    Ep10: val=152c/144x/0v, agn=73c/73x, |D|=0.2909, dR_max=0.499

## Class-IL Evaluation — Structural Advantage Without Task Oracle (§7.3)

IBF natively evaluates the full class space through the corrected coherence landscape — it does not require a task oracle to restrict the candidate set at test time. This gives it a structural Class-IL advantage.

The paper reports: *"IBF reaches 52.8% across all 100 classes without a task oracle, whereas Replay reaches 39.2% and the neural baselines collapse to near chance."*

Baselines are retrained here because the overnight cell's local variables are out of scope.

In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS-IL BASELINE EVALUATION (no task oracle)
# MLP/EWC/Replay evaluated on ALL classes seen so far, no task mask.
# IBF Class-IL data already exists in overnight results.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  CLASS-IL EVALUATION (no task oracle)")
print("=" * 70)

# We need the trained baseline models from the overnight run.
# They should still be in memory from train_ibf's first seed.
# If not, this cell will error — but the overnight cell trained
# baselines only for seed 42's Full IBF run.

# Check what's available
baselines_available = True
for name in ['mlp_m', 'rep_m', 'ewc_m']:
    if name not in dir() or eval(name) is None:
        # Try alternate names from overnight cell scope
        pass

# The overnight cell used local variables inside train_ibf().
# Baselines are lost. We need to retrain them quickly.
# BUT: we saved the accuracy matrices for Task-IL.
# For Class-IL, we need to evaluate without the task mask.
# This requires either saved models or retraining.

# FAST PATH: Retrain baselines (same protocol, ~10 min total)
print("  Retraining baselines for Class-IL evaluation...")

random.seed(42); np.random.seed(42); torch.manual_seed(42)
tsks_42 = build_task_splits(42)

mlp_ci = MLPClassifier(Z_DIM, 128, N_CLASSES).to(DEVICE)
mlp_ci_opt = torch.optim.Adam(mlp_ci.parameters(), lr=1e-3)

rep_ci = ReplayClassifier(Z_DIM, 128, N_CLASSES, 5000).to(DEVICE)
rep_ci_opt = torch.optim.Adam(rep_ci.parameters(), lr=1e-3)

ewc_ci = EWCClassifier(Z_DIM, 128, N_CLASSES, 1000).to(DEVICE)
ewc_ci_opt = torch.optim.Adam(ewc_ci.parameters(), lr=1e-3)

for task_id in range(N_TASKS):
    tc = tsks_42[task_id]
    tidx = np.array([i for c in tc for i in train_indices_by_class[c]])
    tz = z64_train_all[tidx]
    ty = train_labels_all[tidx]
    nt = len(tz)

    for epoch in range(EPOCHS_PER_TASK):
        perm = np.random.permutation(nt)
        for bs in range(0, nt, 64):
            bidx = perm[bs:bs + 64]
            z_b = torch.tensor(tz[bidx]).float().to(DEVICE)
            y_b = torch.tensor(ty[bidx]).long().to(DEVICE)

            mlp_ci.train()
            loss = nn.CrossEntropyLoss()(mlp_ci(z_b), y_b)
            mlp_ci_opt.zero_grad(); loss.backward(); mlp_ci_opt.step()

            rep_ci.train()
            loss_r = nn.CrossEntropyLoss()(rep_ci(z_b), y_b)
            rz, ry = rep_ci.get_replay_batch(64)
            if rz is not None:
                rz_t = torch.tensor(rz).float().to(DEVICE)
                ry_t = torch.tensor(ry).long().to(DEVICE)
                loss_r = loss_r + nn.CrossEntropyLoss()(rep_ci(rz_t), ry_t)
            rep_ci_opt.zero_grad(); loss_r.backward(); rep_ci_opt.step()

            ewc_ci.train()
            loss_e = (nn.CrossEntropyLoss()(ewc_ci(z_b), y_b)
                      + ewc_ci.ewc_penalty())
            ewc_ci_opt.zero_grad(); loss_e.backward(); ewc_ci_opt.step()

        if epoch == EPOCHS_PER_TASK - 1:
            rep_ci.add_to_buffer(tz, ty)

    ewc_ci.compute_fisher(tz, ty)

    if (task_id + 1) % 5 == 0:
        print(f"    Trained through Task {task_id}")

print("  Baselines retrained. Evaluating Class-IL...")

# Evaluate Class-IL: all 100 classes, NO task mask
all_classes = []
for t in range(N_TASKS):
    all_classes.extend(tsks_42[t])

classil_results = {}
with torch.no_grad():
    for name, model in [('MLP', mlp_ci), ('Replay', rep_ci), ('EWC', ewc_ci)]:
        model.eval()
        correct = 0
        total = 0
        for t in range(N_TASKS):
            tc = tsks_42[t]
            eidx = np.array([i for c in tc for i in test_indices_by_class[c]])
            ez = z64_test[eidx]
            ey = test_labels_all[eidx]
            z_t = torch.tensor(ez).float().to(DEVICE)
            # NO task mask — raw 100-way prediction
            preds = model(z_t).argmax(1).cpu().numpy()
            correct += (preds == ey).sum()
            total += len(ey)
        acc = correct / total
        classil_results[name] = acc
        print(f"    {name}: {acc:.4f} ({correct}/{total})")

# IBF Class-IL from overnight results
ibf_ci = ALL_RESULTS['full_42']['avg_classil_log']

print(f"\n{'═' * 70}")
print(f"  CLASS-IL COMPARISON (seed 42, no task oracle)")
print(f"{'═' * 70}")
print(f"  {'Agent':<15s}  {'Class-IL':>10s}  {'Task-IL':>10s}  "
      f"{'Needs oracle':>14s}")
print(f"  {'─' * 55}")
print(f"  {'IBF (log)':<15s}  {ibf_ci:>10.4f}  "
      f"{ALL_RESULTS['full_42']['avg_taskil_log']:>10.4f}  "
      f"{'No':>14s}")
print(f"  {'Coh head':<15s}  {'—':>10s}  {'0.9010':>10s}  "
      f"{'—':>14s}")
print(f"  {'Replay':<15s}  {classil_results['Replay']:>10.4f}  "
      f"{'0.7226':>10s}  {'Yes':>14s}")
print(f"  {'EWC':<15s}  {classil_results['EWC']:>10.4f}  "
      f"{'0.3114':>10s}  {'Yes':>14s}")
print(f"  {'MLP':<15s}  {classil_results['MLP']:>10.4f}  "
      f"{'0.2761':>10s}  {'Yes':>14s}")
print(f"{'═' * 70}")

# Save
classil_out = {
    'ibf_classil_log': ibf_ci,
    'mlp_classil': classil_results['MLP'],
    'replay_classil': classil_results['Replay'],
    'ewc_classil': classil_results['EWC'],
}
with open(os.path.join(OUT_DIR, 'classil_comparison.json'), 'w') as f:
    json.dump(classil_out, f, indent=2)

del mlp_ci, rep_ci, ewc_ci, mlp_ci_opt, rep_ci_opt, ewc_ci_opt

  CLASS-IL EVALUATION (no task oracle)
  Retraining baselines for Class-IL evaluation...
    Trained through Task 4
    Trained through Task 9
    Trained through Task 14
    Trained through Task 19
  Baselines retrained. Evaluating Class-IL...
    MLP: 0.0491 (491/10000)
    Replay: 0.3922 (3922/10000)
    EWC: 0.0520 (520/10000)

══════════════════════════════════════════════════════════════════════
  CLASS-IL COMPARISON (seed 42, no task oracle)
══════════════════════════════════════════════════════════════════════
  Agent              Class-IL     Task-IL    Needs oracle
  ───────────────────────────────────────────────────────
  IBF (log)            0.5276      0.9026              No
  Coh head                  —      0.9010               —
  Replay               0.3922      0.7226             Yes
  EWC                  0.0520      0.3114             Yes
  MLP                  0.0491      0.2761             Yes
══════════════════════════════════════════════════════════════════════

In [42]:
# ── Consolidate all CIFAR results into one JSON ──

cifar_final = {
    'config': {
        'n_tasks': N_TASKS,
        'classes_per_task': CLASSES_PER_TASK,
        'z_dim': Z_DIM,
        'z_dim_aug': Z_DIM_AUG,
        'sigma': float(SIGMA_68D),
        'sigma_agency': float(SIGMA_64D_AGENCY),
        'merge': float(MERGE_68D),
        'kappa': float(SIGMA_68D / np.sqrt(d_eff)),
        'd_eff': float(d_eff),
        'sigma_floor': 1.25,
        'alpha_shrink': 10.0,
        'coh_pretrain_per_class': COH_PRETRAIN_PER_CLASS,
        'coh_baseline_taskil': 0.901,
        'epochs_per_task': EPOCHS_PER_TASK,
    },
    'runs': {k: v for k, v in ALL_RESULTS.items()},
    'classil': {
        'ibf_log': ALL_RESULTS['full_42']['avg_classil_log'],
        'mlp': 0.0491,
        'replay': 0.3922,
        'ewc': 0.0520,
    },
}

with open(os.path.join(OUT_DIR, 'cifar_final.json'), 'w') as f:
    json.dump(cifar_final, f, indent=2)

print(f"Saved: {OUT_DIR}/cifar_final.json")

Saved: /workspace/cifar_out/cifar_final.json
